# Chapter 4: Coupling and State-Space Assumptions

This notebook implements the Chapter 4 experiments directly in notebook cells.  It uses small, tested project utilities from `src/` for reusable primitives such as `VelocityMLP`, Sinkhorn OT, standard metrics, and toy data representation helpers, but the experiment control flow is visible here: each experiment constructs its data, coupling, model, rollout, metrics, figures, and saved artifacts cell by cell.

Claim boundaries used throughout this notebook:

- EB training space is standardized PC-20.
- EB OT cost space is standardized PC-20 unless Exp 7 explicitly constructs a PHATE contrastive diagnostic.
- EB display space is PHATE 2D only for visualization.
- EB endpoint MMD and sliced W2 are always computed in PC-20.
- Couplings are training assumptions, not observed lineage.
- Reflow straightness is a solver/path-geometry diagnostic, not biological validation.


## 0. Setup

Imports, paths, device selection, random seeds, output directories, plotting style, cache helpers, and run-size controls.  The defaults below match the requested chapter settings; environment variables are only for smoke testing or resuming partial runs.


In [ ]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig_ch04")

from pathlib import Path
import sys
import json
import math
import time
import random
import hashlib
import warnings
from dataclasses import dataclass
from typing import Callable, Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import torch
    from torch import nn
except Exception as exc:
    raise ImportError("Chapter 4 experiments require PyTorch.") from exc

try:
    import anndata as ad
except Exception:
    ad = None

from scipy import sparse
from scipy.sparse.csgraph import shortest_path
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path("/home/xmabs/flow_matching_for_dynamic_biology/flow_matching_for_dynamic_biology")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models import VelocityMLP as ProjectVelocityMLP, count_parameters
from src.losses import cfm_batch, cfm_loss_from_pairs
from src.train import train_cfm_steps
from src.sampling import euler_sample
from src.samplers import CouplingPairSampler as SrcCouplingPairSampler
from src.ot import (
    pairwise_squared_distances,
    independent_coupling,
    compute_ot_coupling_from_cost,
    sample_pair_indices_from_coupling,
    coupling_diagnostics,
)
from src.metrics import (
    mmd_rbf as project_mmd_rbf,
    sliced_wasserstein_distance,
    endpoint_metrics,
    trajectory_path_length,
    trajectory_path_energy,
    trajectory_straightness,
    off_manifold_knn_distance,
    fate_mass_error,
    coupling_l1_distance,
    normalized_cost_matrix,
    distribution_readout_metrics,
)
from src.representations import (
    fit_pca_state_space,
    pca_inverse_transform,
    program_index_dict,
    readout_program_scores_from_matrix,
    standardize_train_space,
    nearest_neighbor_overlap,
)

SEEDS = [42, 43, 44]
DEFAULT_SEED = 42
SOURCE_TIME = "1"
TARGET_TIME = "2"
TRAINING_STEPS = int(os.environ.get("CH04_TRAINING_STEPS", "1500"))
BATCH_SIZE = int(os.environ.get("CH04_BATCH_SIZE", "256"))
DEFAULT_NFE = int(os.environ.get("CH04_DEFAULT_NFE", "64"))
NFE_GRID = [2, 4, 8, 16, 32, 64]
SINKHORN_EPSILON = float(os.environ.get("CH04_SINKHORN_EPSILON", "0.05"))
EPSILON_GRID = [0.01, 0.02, 0.05, 0.1, 0.5]
BOOTSTRAP_REPEATS = int(os.environ.get("CH04_BOOTSTRAP_REPEATS", "50"))
EB_MAX_CELLS_PER_TIME = os.environ.get("CH04_EB_MAX_CELLS_PER_TIME", "")
EB_MAX_CELLS_PER_TIME = None if EB_MAX_CELLS_PER_TIME == "" else int(EB_MAX_CELLS_PER_TIME)
TOY_TRAINING_STEPS = int(os.environ.get("CH04_TOY_TRAINING_STEPS", str(TRAINING_STEPS)))
SMOKE_MODE = os.environ.get("CH04_SMOKE_MODE", "0") == "1"
if SMOKE_MODE:
    TRAINING_STEPS = min(TRAINING_STEPS, 20)
    TOY_TRAINING_STEPS = min(TOY_TRAINING_STEPS, 20)
    BATCH_SIZE = min(BATCH_SIZE, 64)
    DEFAULT_NFE = min(DEFAULT_NFE, 8)
    NFE_GRID = [2, 4, 8]
    BOOTSTRAP_REPEATS = min(BOOTSTRAP_REPEATS, 3)
    EB_MAX_CELLS_PER_TIME = 120 if EB_MAX_CELLS_PER_TIME is None else min(EB_MAX_CELLS_PER_TIME, 120)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = PROJECT_ROOT / "data"
EB_PATH = DATA_DIR / "trajectorynet_eb" / "eb_velocity_v5.npz"
TOY_DIR = DATA_DIR / "toy_branching_snapshots"
TOY_CSV_PATH = TOY_DIR / "observed_2d_snapshots.csv"
TOY_H5AD_PATH = TOY_DIR / "branching_toy_pseudocounts.h5ad"

FIG_DIR = PROJECT_ROOT /  "figures" / "ch04"
OUT_DIR = PROJECT_ROOT /  "outputs" / "ch04"
if SMOKE_MODE:
    FIG_DIR = FIG_DIR / "smoke"
    OUT_DIR = OUT_DIR / "smoke"
CACHE_DIR = OUT_DIR / "cache"
for path in [FIG_DIR, OUT_DIR, CACHE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 220,
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "legend.fontsize": 8,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

PALETTE = {
    "source": "#4C78A8",
    "target": "#F58518",
    "random": "#8E8E8E",
    "ot": "#008A7A",
    "reflow1": "#5369A6",
    "reflow2": "#B279A2",
    "rare": "#D95F02",
    "major": "#2C7FB8",
    "program": "#54A24B",
    "diagnostic": "#E45756",
}

print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {DEVICE}")
print(f"Training steps: {TRAINING_STEPS}; batch size: {BATCH_SIZE}; default NFE: {DEFAULT_NFE}")
print(f"Smoke mode: {SMOKE_MODE}")


In [ ]:
def set_seed(seed: int = DEFAULT_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def json_ready(obj):
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, pd.DataFrame):
        return obj.to_dict(orient="records")
    if isinstance(obj, dict):
        return {str(k): json_ready(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_ready(v) for v in obj]
    return obj


def save_json(path: str | Path, payload: dict) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(json_ready(payload), indent=2, sort_keys=True))
    return path


def load_json(path: str | Path):
    return json.loads(Path(path).read_text())


def save_npz(path: str | Path, **arrays) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(path, **arrays)
    return path


def load_npz(path: str | Path):
    return np.load(Path(path), allow_pickle=True)


def save_csv(path: str | Path, frame: pd.DataFrame) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)
    return path


def save_pt(path: str | Path, payload) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(payload, path)
    return path


def load_pt(path: str | Path, map_location=None):
    return torch.load(Path(path), map_location=map_location or DEVICE)


def save_figure(fig, filename: str, close: bool = True) -> Path:
    path = FIG_DIR / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    if close:
        plt.close(fig)
    return path


def artifact_exists(*paths) -> bool:
    return all(Path(p).exists() and Path(p).stat().st_size > 0 for p in paths)


def stable_hash(*items) -> str:
    h = hashlib.sha1()
    for item in items:
        h.update(str(item).encode())
    return h.hexdigest()[:10]


def sample_rows(n: int, max_n: int | None, seed: int = DEFAULT_SEED) -> np.ndarray:
    idx = np.arange(int(n))
    if max_n is None or n <= max_n:
        return idx
    rng = np.random.default_rng(seed)
    return np.sort(rng.choice(idx, size=int(max_n), replace=False))


def as_float32(x):
    return np.asarray(x, dtype=np.float32)


def to_tensor(x, device: torch.device = DEVICE):
    return torch.as_tensor(x, dtype=torch.float32, device=device)


def ensure_finite(name: str, x: np.ndarray) -> None:
    if not np.all(np.isfinite(x)):
        raise ValueError(f"{name} contains non-finite values")


## 1. Shared Utilities

The cells below define chapter-level wrappers and helpers.  They intentionally keep the key CFM/Sinkhorn/rollout/metric logic visible while reusing tested primitives from `src` where available.


In [ ]:
class VelocityMLP(ProjectVelocityMLP):
    """Chapter 4 VelocityMLP wrapper with requested defaults.

    The underlying implementation is imported from `src.models`, but the
    requested architecture is fixed here: hidden=128 and layers=4.
    """
    def __init__(self, input_dim: int, hidden: int = 128, layers: int = 4):
        super().__init__(x_dim=int(input_dim), hidden_dim=int(hidden), hidden_layers=int(layers))


def make_time_batch(batch_size: int, device: torch.device = DEVICE) -> torch.Tensor:
    return torch.rand(int(batch_size), 1, device=device)


def sample_independent_pairs(X0, X1, n_pairs: int, seed: int = DEFAULT_SEED):
    rng = np.random.default_rng(seed)
    i0 = rng.integers(0, len(X0), size=int(n_pairs))
    i1 = rng.integers(0, len(X1), size=int(n_pairs))
    return i0, i1


def compute_cost_matrix(x0, x1, normalize: bool = True):
    C = pairwise_squared_distances(np.asarray(x0, dtype=np.float32), np.asarray(x1, dtype=np.float32)).astype(np.float32)
    if not normalize:
        return C, 1.0
    positive = C[C > 0]
    scale = float(np.median(positive)) if positive.size else 1.0
    scale = max(scale, 1e-12)
    return (C / scale).astype(np.float32), scale


def sinkhorn_plan(C, epsilon: float = SINKHORN_EPSILON, return_info: bool = False):
    """Balanced Sinkhorn plan using POT when available, with project fallback."""
    return compute_ot_coupling_from_cost(np.asarray(C, dtype=np.float32), epsilon=float(epsilon), return_info=return_info)


def sample_from_plan(pi, n_pairs: int, seed: int = DEFAULT_SEED):
    return sample_pair_indices_from_coupling(np.asarray(pi, dtype=float), batch_size=int(n_pairs), seed=seed)


class PlanPairSampler:
    """Notebook-local thin wrapper around `src.samplers.CouplingPairSampler`."""
    def __init__(self, X0, X1, pi=None, seed: int = DEFAULT_SEED, labels0=None, labels1=None):
        self.X0 = as_float32(X0)
        self.X1 = as_float32(X1)
        if pi is None:
            pi = independent_coupling(len(self.X0), len(self.X1))
        self.pi = np.asarray(pi, dtype=float)
        self.src_sampler = SrcCouplingPairSampler(self.X0, self.X1, self.pi, seed=seed, labels0=labels0, labels1=labels1)

    def sample(self, batch_size: int = BATCH_SIZE):
        return self.src_sampler.sample(int(batch_size))


def train_cfm(
    model,
    pair_sampler,
    steps: int = TRAINING_STEPS,
    batch_size: int = BATCH_SIZE,
    lr: float = 1e-3,
    device: torch.device = DEVICE,
    seed: int = DEFAULT_SEED,
    log_every: int = 250,
):
    """Thin notebook wrapper around `src.train.train_cfm_steps`."""
    set_seed(seed)
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=float(lr))
    history = train_cfm_steps(
        model,
        pair_sampler.sample,
        optimizer,
        n_steps=int(steps),
        batch_size=int(batch_size),
        device=device,
        log_every=int(log_every),
    )
    return history


@torch.no_grad()
def rollout_euler(model, x0, nfe: int = DEFAULT_NFE, device: torch.device = DEVICE):
    """Thin notebook wrapper around `src.sampling.euler_sample`."""
    model.eval()
    x0_t = to_tensor(x0, device)
    endpoint, _, _ = euler_sample(model, x0_t, n_steps=int(nfe), return_traj=False)
    return endpoint.detach().cpu().numpy().astype(np.float32)


@torch.no_grad()
def trajectory_rollout(model, x0, nfe: int = DEFAULT_NFE, return_path: bool = True, device: torch.device = DEVICE):
    """Thin notebook wrapper around `src.sampling.euler_sample`."""
    model.eval()
    x0_t = to_tensor(x0, device)
    endpoint, traj_t, _ = euler_sample(model, x0_t, n_steps=int(nfe), return_traj=return_path)
    endpoint_np = endpoint.detach().cpu().numpy().astype(np.float32)
    if return_path:
        traj_np = traj_t.detach().cpu().numpy().astype(np.float32)
        return endpoint_np, traj_np, np.linspace(0.0, 1.0, int(nfe) + 1)
    return endpoint_np


def mmd_rbf(X, Y, gamma: float | None = None):
    return project_mmd_rbf(np.asarray(X, dtype=float), np.asarray(Y, dtype=float), gamma=gamma)


def sliced_w2(X, Y, n_projections: int = 128, seed: int = DEFAULT_SEED):
    return sliced_wasserstein_distance(X, Y, n_projections=n_projections, seed=seed)


def path_length(traj):
    return trajectory_path_length(np.asarray(traj, dtype=float))


def path_energy(traj, times=None):
    return trajectory_path_energy(np.asarray(traj, dtype=float), times=times)


def tortuosity_straightness(traj):
    """Path-length/end-point tortuosity minus one; geometric, not biological validation."""
    return trajectory_straightness(np.asarray(traj, dtype=float))


def straightness(traj):
    """Backward-compatible alias used only inside this notebook."""
    return tortuosity_straightness(traj)


def straightness_action_S(traj, times=None):
    """Planning straightness action S = integral E[||(Z1-Z0)-dZ_t/dt||^2] dt.

    Euler trajectories provide finite-difference velocities. Lower S means
    finite-difference path velocities are closer to the endpoint chord.
    """
    traj = np.asarray(traj, dtype=float)
    if traj.ndim != 3 or traj.shape[0] < 2:
        raise ValueError("traj must have shape (T, N, D) with T >= 2")
    if times is None:
        times = np.linspace(0.0, 1.0, traj.shape[0])
    times = np.asarray(times, dtype=float)
    dt = np.diff(times)
    if len(dt) != traj.shape[0] - 1 or np.any(dt <= 0):
        raise ValueError("times must be strictly increasing and match traj")
    vel = np.diff(traj, axis=0) / dt[:, None, None]
    chord = traj[-1] - traj[0]
    sq = np.sum((chord[None, :, :] - vel) ** 2, axis=-1)
    return float(np.sum(sq.mean(axis=1) * dt))


def off_manifold_knn(points, reference, k: int = 15, batch_size: int = 1000):
    points_arr = np.asarray(points, dtype=np.float32)
    reference_arr = np.asarray(reference, dtype=np.float32)
    if points_arr.ndim == 3:
        points_arr = points_arr.reshape(-1, points_arr.shape[-1])
    if points_arr.ndim != 2 or reference_arr.ndim != 2:
        raise ValueError("points and reference must be 2D arrays, or points may be a (T, N, D) trajectory")
    if points_arr.shape[1] != reference_arr.shape[1]:
        raise ValueError("points and reference must have the same feature dimension")
    if points_arr.shape[0] <= batch_size:
        return off_manifold_knn_distance(points_arr, reference_arr, k=k)
    from sklearn.neighbors import NearestNeighbors
    kk = max(1, min(int(k), reference_arr.shape[0]))
    nn = NearestNeighbors(n_neighbors=kk, algorithm="ball_tree", leaf_size=40, n_jobs=1)
    nn.fit(reference_arr)
    total = 0.0
    count = 0
    for start in range(0, points_arr.shape[0], int(batch_size)):
        distances, _ = nn.kneighbors(points_arr[start:start + int(batch_size)])
        total += float(distances.sum())
        count += int(distances.size)
    return total / max(count, 1)

def coarse_step_error(model, x0, nfe_coarse: int = 4, nfe_fine: int = 64):
    z_coarse = rollout_euler(model, x0, nfe=nfe_coarse)
    z_fine = rollout_euler(model, x0, nfe=nfe_fine)
    return float(np.linalg.norm(z_coarse - z_fine, axis=1).mean())


def effective_support(pi):
    pi = np.asarray(pi, dtype=float)
    p = pi[pi > 0]
    return float(np.exp(-np.sum(p * np.log(p)))) if p.size else 0.0


def plan_entropy(pi):
    pi = np.asarray(pi, dtype=float)
    p = pi[pi > 0]
    return -float(np.sum(p * np.log(p))) if p.size else 0.0


def topk_nn_overlap(X_a, X_b, k: int = 15):
    return nearest_neighbor_overlap(X_a, X_b, k=k)


def coupling_topk_overlap(pi_a, pi_b, k: int = 15):
    pi_a = np.asarray(pi_a, dtype=float)
    pi_b = np.asarray(pi_b, dtype=float)
    if pi_a.shape != pi_b.shape:
        raise ValueError("coupling shapes differ")
    k = max(1, min(int(k), pi_a.shape[1]))
    rows = []
    for a, b in zip(pi_a, pi_b):
        ta = set(np.argpartition(-a, kth=k - 1)[:k].tolist())
        tb = set(np.argpartition(-b, kth=k - 1)[:k].tolist())
        rows.append(len(ta & tb) / float(k))
    return float(np.mean(rows))


def evaluate_endpoint(pred, target, seed: int = DEFAULT_SEED):
    return {
        "endpoint_mmd": mmd_rbf(pred, target),
        "sliced_w2": sliced_w2(pred, target, seed=seed),
    }


In [ ]:
def plot_phate_pairs(X0_phate, X1_phate, idx0, idx1, title: str, max_lines: int = 120, seed: int = DEFAULT_SEED, values=None):
    return plot_pair_panels(
        X0_phate,
        X1_phate,
        [{"title": title, "idx0": idx0, "idx1": idx1, "values": values, "seed": seed}],
        filename="_temporary_pair_panel.png",
        title=title,
    )


def plot_pair_panels(X0_phate, X1_phate, panels, filename: str, title: str, value_label: str = "PC-20 chord length"):
    from matplotlib.collections import LineCollection
    from matplotlib.colors import Normalize
    from matplotlib.cm import ScalarMappable

    n = len(panels)
    fig, axes = plt.subplots(1, n, figsize=(5.0 * n, 4.2), squeeze=False)
    axes_flat = axes[0]
    all_values = []
    for panel in panels:
        if panel.get("values") is not None:
            all_values.append(np.asarray(panel["values"], dtype=float))
    norm = None
    if all_values:
        finite = np.concatenate([v[np.isfinite(v)] for v in all_values if np.any(np.isfinite(v))])
        norm = Normalize(vmin=float(finite.min()), vmax=float(finite.max())) if finite.size else None
    for ax, panel in zip(axes_flat, panels):
        idx0, idx1 = panel["idx0"], panel["idx1"]
        ax.scatter(X0_phate[:, 0], X0_phate[:, 1], s=8, color=PALETTE["source"], alpha=0.20, linewidths=0)
        ax.scatter(X1_phate[:, 0], X1_phate[:, 1], s=8, color=PALETTE["target"], alpha=0.20, linewidths=0)
        keep = sample_rows(len(idx0), min(panel.get("max_lines", 100), len(idx0)), seed=panel.get("seed", DEFAULT_SEED))
        segments = np.stack([X0_phate[idx0[keep]], X1_phate[idx1[keep]]], axis=1)
        values = panel.get("values")
        if values is not None and norm is not None:
            lc = LineCollection(segments, cmap="viridis", norm=norm, linewidths=0.8, alpha=0.55)
            lc.set_array(np.asarray(values, dtype=float)[keep])
            ax.add_collection(lc)
        else:
            lc = LineCollection(segments, colors=panel.get("color", "0.45"), linewidths=0.7, alpha=0.25)
            ax.add_collection(lc)
        ax.set_title(panel["title"])
        ax.set_xlabel("PHATE 1")
        ax.set_ylabel("PHATE 2")
    if norm is not None:
        fig.colorbar(ScalarMappable(norm=norm, cmap="viridis"), ax=list(axes_flat), fraction=0.035, pad=0.02, label=value_label)
    fig.suptitle(title, y=1.02)
    return save_figure(fig, filename)


def add_local_arrows(ax, projected_traj, observed_phate, color, max_arrows: int = 28, seed: int = DEFAULT_SEED):
    """Draw local average arrows only for trajectory points close to observed PHATE neighborhoods."""
    projected_traj = np.asarray(projected_traj, dtype=float)
    observed_phate = np.asarray(observed_phate, dtype=float)
    if projected_traj.shape[0] < 3 or projected_traj.shape[-1] != 2:
        return
    mid_step = projected_traj.shape[0] // 2
    anchors = projected_traj[mid_step]
    deltas = projected_traj[min(mid_step + 1, projected_traj.shape[0] - 1)] - projected_traj[max(mid_step - 1, 0)]
    nn_obs = NearestNeighbors(n_neighbors=min(15, len(observed_phate))).fit(observed_phate)
    obs_r = nn_obs.kneighbors(observed_phate, return_distance=True)[0][:, -1]
    dist = nn_obs.kneighbors(anchors, return_distance=True)[0][:, 0]
    threshold = float(np.quantile(obs_r, 0.75))
    valid = np.flatnonzero(dist <= threshold)
    if valid.size == 0:
        return
    keep = sample_rows(len(valid), min(max_arrows, len(valid)), seed=seed)
    valid = valid[keep]
    ax.quiver(
        anchors[valid, 0], anchors[valid, 1], deltas[valid, 0], deltas[valid, 1],
        angles="xy", scale_units="xy", scale=1.0, width=0.004, color=color, alpha=0.75,
    )


def plot_projected_trajectories(paths, X0_phate, X1_phate, pc_to_phate, filename: str, title: str, max_lines: int = 45, local_arrows: bool = True):
    n = len(paths)
    fig, axes = plt.subplots(1, n, figsize=(5.0 * n, 4.2), squeeze=False)
    observed_phate = np.vstack([X0_phate, X1_phate])
    for ax, (name, traj) in zip(axes[0], paths.items()):
        ax.scatter(X0_phate[:, 0], X0_phate[:, 1], s=8, color=PALETTE["source"], alpha=0.18, linewidths=0)
        ax.scatter(X1_phate[:, 0], X1_phate[:, 1], s=8, color=PALETTE["target"], alpha=0.16, linewidths=0)
        keep = sample_rows(traj.shape[1], min(max_lines, traj.shape[1]), seed=DEFAULT_SEED)
        selected = np.asarray(traj[:, keep, :], dtype=np.float32)
        flat = selected.reshape(-1, selected.shape[-1])
        ph = pc_to_phate(flat).reshape(selected.shape[0], selected.shape[1], 2)
        color = PALETTE.get(name, "0.35")
        for j in range(ph.shape[1]):
            ax.plot(ph[:, j, 0], ph[:, j, 1], color=color, alpha=0.55, linewidth=1.0)
        if local_arrows:
            add_local_arrows(ax, ph, observed_phate, color=color, seed=DEFAULT_SEED + 7)
        ax.set_title(name.replace("_", " "))
        ax.set_xlabel("PHATE 1 (display only)")
        ax.set_ylabel("PHATE 2 (display only)")
    fig.suptitle(title, y=1.02)
    return save_figure(fig, filename)


def plot_metric_lines(table, x: str, y: str, hue: str, filename: str, title: str):
    fig, ax = plt.subplots(figsize=(6.0, 4.0))
    for name, group in table.groupby(hue):
        group = group.sort_values(x)
        ax.plot(group[x], group[y], marker="o", linewidth=1.5, label=str(name).replace("_", " "))
    ax.set_xscale("log", base=2 if set(table[x]).issubset(set(NFE_GRID)) else 10)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(title)
    ax.legend(frameon=False)
    return save_figure(fig, filename)


def plot_heatmap(matrix, title: str, filename: str, max_size: int = 120, cmap: str = "viridis"):
    M = np.asarray(matrix)
    rows = sample_rows(M.shape[0], min(max_size, M.shape[0]), seed=DEFAULT_SEED)
    cols = sample_rows(M.shape[1], min(max_size, M.shape[1]), seed=DEFAULT_SEED + 1)
    fig, ax = plt.subplots(figsize=(4.8, 4.2))
    im = ax.imshow(M[np.ix_(rows, cols)], aspect="auto", cmap=cmap)
    ax.set_title(title)
    ax.set_xlabel("target subset")
    ax.set_ylabel("source subset")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    return save_figure(fig, filename)


def plot_table_image(frame: pd.DataFrame, filename: str, title: str, max_rows: int = 12):
    shown = frame.head(max_rows).copy()
    fig, ax = plt.subplots(figsize=(min(14, 1.5 + 1.4 * shown.shape[1]), 0.8 + 0.35 * len(shown)))
    ax.axis("off")
    ax.set_title(title, loc="left")
    tbl = ax.table(cellText=shown.round(4).astype(str).values, colLabels=shown.columns, loc="center", cellLoc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(7)
    tbl.scale(1.0, 1.25)
    return save_figure(fig, filename)


## 2. Load EB Data

EB `pcs` is loaded from `eb_velocity_v5.npz`.  This file stores 100 PCs, so this notebook explicitly takes the first 20 PCs and standardizes them using all EB snapshots.  `phate` is kept as display-only coordinates.  The main EB bridge is time label `1 -> 2`.


In [ ]:
def load_eb_data(path: Path = EB_PATH, source_time: str = SOURCE_TIME, target_time: str = TARGET_TIME):
    if not path.exists():
        raise FileNotFoundError(path)
    z = np.load(path, allow_pickle=True)
    keys = list(z.files)
    pcs_full = np.asarray(z["pcs"], dtype=np.float32)
    phate = np.asarray(z["phate"], dtype=np.float32)[:, :2]
    labels = np.asarray(z["sample_labels"]).astype(str)
    pcs20_raw = pcs_full[:, :20].astype(np.float32)
    mean = pcs20_raw.mean(axis=0)
    std = pcs20_raw.std(axis=0)
    std = np.where(std < 1e-6, 1.0, std)
    pcs20 = ((pcs20_raw - mean) / std).astype(np.float32)
    available = sorted(np.unique(labels).tolist(), key=lambda s: int(s) if str(s).isdigit() else str(s))
    if str(source_time) not in available or str(target_time) not in available:
        raise ValueError(f"Requested bridge {source_time}->{target_time}; available labels: {available}")
    idx_source_all = np.flatnonzero(labels == str(source_time))
    idx_target_all = np.flatnonzero(labels == str(target_time))
    if EB_MAX_CELLS_PER_TIME is not None:
        idx_source = idx_source_all[sample_rows(len(idx_source_all), EB_MAX_CELLS_PER_TIME, seed=DEFAULT_SEED)]
        idx_target = idx_target_all[sample_rows(len(idx_target_all), EB_MAX_CELLS_PER_TIME, seed=DEFAULT_SEED + 1)]
    else:
        idx_source, idx_target = idx_source_all, idx_target_all
    counts = pd.Series(labels, name="time").value_counts().sort_index().rename_axis("time").reset_index(name="n_cells")
    summary = {
        "source_path": str(path),
        "available_keys": keys,
        "pcs_shape_actual": list(pcs_full.shape),
        "pc20_shape_used": list(pcs20.shape),
        "phate_shape": list(phate.shape),
        "sample_label_values": counts.to_dict(orient="records"),
        "source_time": str(source_time),
        "target_time": str(target_time),
        "n_source_full": int(len(idx_source_all)),
        "n_target_full": int(len(idx_target_all)),
        "n_source_used": int(len(idx_source)),
        "n_target_used": int(len(idx_target)),
        "training_space": "standardized PC-20 from pcs[:, :20]",
        "ot_cost_space": "standardized PC-20 unless Exp 7 contrastive diagnostic",
        "display_space": "PHATE 2D only for visualization",
        "distributional_metrics_space": "standardized PC-20",
        "standardization": "mean/std fit on all EB snapshots in PC-20",
        "adaptation_note": "Input pcs has 100 columns; this chapter uses the first 20 PCs as PC-20.",
    }
    save_json(OUT_DIR / "eb_data_summary.json", summary)
    return {
        "keys": keys,
        "pcs20_all": pcs20,
        "pcs20_raw_all": pcs20_raw,
        "phate_all": phate,
        "labels": labels,
        "counts": counts,
        "pc_mean": mean,
        "pc_std": std,
        "idx_source": idx_source,
        "idx_target": idx_target,
        "X0_pc": pcs20[idx_source],
        "X1_pc": pcs20[idx_target],
        "X0_phate": phate[idx_source],
        "X1_phate": phate[idx_target],
        "source_time": str(source_time),
        "target_time": str(target_time),
        "summary": summary,
    }

EB = load_eb_data()
print("EB keys:", EB["keys"])
print(EB["counts"])
print("Source/target PC shapes:", EB["X0_pc"].shape, EB["X1_pc"].shape)
print("Summary saved to", OUT_DIR / "eb_data_summary.json")


In [ ]:
# Display-only mapping from PC-20 trajectory points back to PHATE for plotting.
# This is not used for training or endpoint distributional metrics.
pc_to_phate_knn = KNeighborsClassifier(n_neighbors=min(15, len(EB["pcs20_all"])), weights="distance")
pc_to_phate_knn.fit(EB["pcs20_all"], np.arange(len(EB["pcs20_all"])))

def pc_to_phate(points_pc):
    points_pc = np.asarray(points_pc, dtype=np.float32)
    dist, ind = pc_to_phate_knn.kneighbors(points_pc, return_distance=True)
    w = 1.0 / np.clip(dist, 1e-6, None)
    w = w / w.sum(axis=1, keepdims=True)
    return np.einsum("nk,nkd->nd", w, EB["phate_all"][ind])

off_manifold_reference_pc = EB["pcs20_all"]
off_manifold_reference_note = "all available EB snapshots in standardized PC-20"
print(off_manifold_reference_note, off_manifold_reference_pc.shape)


## Exp 1. Independent vs OT Coupling on EB

Train random-product CFM and PC-20 OT-CFM with the same `VelocityMLP(input_dim, hidden=128, layers=4)`, Adam lr `1e-3`, 1500 steps, batch size 256, then evaluate rollouts from source cells against the real target distribution in PC-20.


In [ ]:
X0_eb, X1_eb = EB["X0_pc"], EB["X1_pc"]
X0p_eb, X1p_eb = EB["X0_phate"], EB["X1_phate"]
C_eb_norm, C_eb_scale = compute_cost_matrix(X0_eb, X1_eb, normalize=True)
pi_eb_ot, info_eb_ot = sinkhorn_plan(C_eb_norm, epsilon=SINKHORN_EPSILON, return_info=True)
pi_eb_random = independent_coupling(len(X0_eb), len(X1_eb))
save_npz(CACHE_DIR / "exp1_eb_couplings.npz", C_eb_norm=C_eb_norm, pi_ot=pi_eb_ot, pi_random=pi_eb_random, cost_scale=C_eb_scale)
save_json(CACHE_DIR / "exp1_sinkhorn_info.json", info_eb_ot)
print("OT Sinkhorn info:", info_eb_ot)


In [ ]:
def train_or_load_model(name: str, X0, X1, pi, steps: int = TRAINING_STEPS, seed: int = DEFAULT_SEED):
    cache_tag = f"d{X0.shape[1]}_steps{int(steps)}_batch{BATCH_SIZE}_seed{int(seed)}"
    ckpt = CACHE_DIR / f"{name}_{cache_tag}_model.pt"
    hist_path = CACHE_DIR / f"{name}_{cache_tag}_history.csv"
    model = VelocityMLP(input_dim=X0.shape[1], hidden=128, layers=4).to(DEVICE)
    if ckpt.exists():
        payload = load_pt(ckpt, map_location=DEVICE)
        if int(payload.get("input_dim", -1)) != int(X0.shape[1]) or int(payload.get("steps", -1)) != int(steps):
            raise ValueError(f"Checkpoint metadata mismatch for {ckpt}")
        model.load_state_dict(payload["state_dict"])
        history = pd.read_csv(hist_path) if hist_path.exists() else pd.DataFrame()
        print(f"Loaded {name} from {ckpt}")
        return model, history
    sampler = PlanPairSampler(X0, X1, pi=pi, seed=seed)
    history = train_cfm(model, sampler, steps=steps, batch_size=BATCH_SIZE, lr=1e-3, device=DEVICE, seed=seed)
    save_pt(ckpt, {"state_dict": model.state_dict(), "input_dim": int(X0.shape[1]), "seed": int(seed), "steps": int(steps), "batch_size": int(BATCH_SIZE)})
    save_csv(hist_path, history)
    print(f"Trained {name}; parameters={count_parameters(model)}; final loss={history.loss.iloc[-1]:.4f}")
    return model, history

random_model, random_hist = train_or_load_model("exp1_random_cfm", X0_eb, X1_eb, pi_eb_random, seed=42)
ot_model, ot_hist = train_or_load_model("exp1_ot_cfm", X0_eb, X1_eb, pi_eb_ot, seed=42)


In [ ]:
def midpoint_direction_dispersion(X0, X1, pi, n_pairs: int = 4096, k: int = 25, seed: int = DEFAULT_SEED):
    """Neighborhood dispersion of sampled chord directions around pair midpoints."""
    idx0, idx1 = sample_from_plan(pi, n_pairs, seed=seed)
    x0, x1 = X0[idx0], X1[idx1]
    chords = x1 - x0
    chord_norm = np.linalg.norm(chords, axis=1)
    directions = chords / np.clip(chord_norm[:, None], 1e-12, None)
    midpoints = 0.5 * (x0 + x1)
    n_neighbors = max(2, min(int(k), len(midpoints)))
    nn = NearestNeighbors(n_neighbors=n_neighbors).fit(midpoints)
    neigh = nn.kneighbors(midpoints, return_distance=False)
    angular_std = []
    for ids in neigh:
        local_dirs = directions[ids]
        mean_dir = local_dirs.mean(axis=0)
        mean_norm = np.linalg.norm(mean_dir)
        if mean_norm < 1e-12:
            angles = np.full(len(ids), 90.0)
        else:
            mean_dir = mean_dir / mean_norm
            cosang = np.clip(local_dirs @ mean_dir, -1.0, 1.0)
            angles = np.degrees(np.arccos(cosang))
        angular_std.append(float(np.std(angles)))
    stats = pd.DataFrame({
        "idx0": idx0,
        "idx1": idx1,
        "pc20_chord_length": chord_norm,
        "midpoint_direction_angular_std_deg": angular_std,
    })
    return stats

exp1_rows = []
exp1_paths = {}
pair_stats_exp1 = {}
for name, model, pi in [("random_cfm", random_model, pi_eb_random), ("ot_cfm", ot_model, pi_eb_ot)]:
    z, traj, times = trajectory_rollout(model, X0_eb, nfe=DEFAULT_NFE, return_path=True)
    exp1_paths[name] = traj
    metrics = evaluate_endpoint(z, X1_eb)
    pair_stats = midpoint_direction_dispersion(X0_eb, X1_eb, pi, n_pairs=4096, k=25, seed=44)
    pair_stats["method"] = name
    pair_stats_exp1[name] = pair_stats
    chord = pair_stats["pc20_chord_length"].to_numpy()
    disp = pair_stats["midpoint_direction_angular_std_deg"].to_numpy()
    row = {
        "method": name,
        **metrics,
        "mean_pc20_chord_length": float(chord.mean()),
        "median_pc20_chord_length": float(np.median(chord)),
        "std_pc20_chord_length": float(chord.std()),
        "midpoint_direction_angular_std_mean_deg": float(disp.mean()),
        "midpoint_direction_angular_std_median_deg": float(np.median(disp)),
        "midpoint_direction_angular_std_iqr_deg": float(np.quantile(disp, 0.75) - np.quantile(disp, 0.25)),
        "training_space": "standardized PC-20",
        "cost_space": "standardized PC-20" if name == "ot_cfm" else "independent product coupling",
        "display_space": "PHATE 2D only for figures",
        "endpoint_metric_space": "standardized PC-20",
    }
    exp1_rows.append(row)
    save_npz(CACHE_DIR / f"exp1_{name}_rollout.npz", endpoint=z, trajectory=traj, times=times)

exp1_pair_stats = pd.concat(pair_stats_exp1.values(), ignore_index=True)
save_csv(CACHE_DIR / "exp1_pair_level_statistics.csv", exp1_pair_stats)
exp1_metrics = pd.DataFrame(exp1_rows)
save_csv(OUT_DIR / "exp1_metrics.csv", exp1_metrics)
exp1_metrics


In [ ]:
# Figures requested for Exp 1.
idx_r0, idx_r1 = sample_from_plan(pi_eb_random, 900, seed=50)
idx_o0, idx_o1 = sample_from_plan(pi_eb_ot, 900, seed=51)
r_chord = np.linalg.norm(X1_eb[idx_r1] - X0_eb[idx_r0], axis=1)
o_chord = np.linalg.norm(X1_eb[idx_o1] - X0_eb[idx_o0], axis=1)
plot_pair_panels(
    X0p_eb,
    X1p_eb,
    [
        {"title": "Random product pairs", "idx0": idx_r0, "idx1": idx_r1, "values": r_chord, "seed": 50, "max_lines": 140},
        {"title": "PC-20 OT pairs", "idx0": idx_o0, "idx1": idx_o1, "values": o_chord, "seed": 51, "max_lines": 140},
    ],
    "fig4_2_random_vs_ot_pairs.png",
    "Endpoint pair choices displayed in PHATE; line color is PC-20 chord length",
    value_label="standardized PC-20 chord length",
)

fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.0))
for method, color in [("random_cfm", PALETTE["random"]), ("ot_cfm", PALETTE["ot"] )]:
    stats = pair_stats_exp1[method]
    axes[0].hist(stats["pc20_chord_length"], bins=45, alpha=0.55, color=color, label=method.replace("_", " "))
axes[0].set_title("PC-20 chord length")
axes[0].set_xlabel("standardized PC-20 distance")
axes[0].set_ylabel("sampled pairs")
axes[0].legend(frameon=False)

box_data = [
    pair_stats_exp1["random_cfm"]["midpoint_direction_angular_std_deg"].to_numpy(),
    pair_stats_exp1["ot_cfm"]["midpoint_direction_angular_std_deg"].to_numpy(),
]
axes[1].boxplot(box_data, labels=["random", "OT"], showfliers=False)
axes[1].set_title("Midpoint-neighborhood direction dispersion")
axes[1].set_ylabel("angular std around local mean direction (deg)")
save_figure(fig, "fig4_1_supp_pair_statistics.png")

plot_projected_trajectories(
    {"random": exp1_paths["random_cfm"], "ot": exp1_paths["ot_cfm"]},
    X0p_eb,
    X1p_eb,
    pc_to_phate,
    "fig4_5_random_vs_ot_projected_trajectories.png",
    "Euler rollouts projected to PHATE with local neighborhood arrows",
)
plot_projected_trajectories(
    {"random": exp1_paths["random_cfm"], "ot": exp1_paths["ot_cfm"]},
    X0p_eb,
    X1p_eb,
    pc_to_phate,
    "fig4_1_independent_coupling_paths.png",
    "Independent versus OT-CFM paths (PHATE display only)",
)


## Exp 2. Sinkhorn Epsilon + Minibatch Ablation

Part A varies Sinkhorn epsilon on the median-normalized PC-20 cost.  Part B estimates minibatch-coupling variability by repeatedly recomputing Sinkhorn on random source/target subsets.  PHATE pair distances are display diagnostics only.


In [ ]:
eps_rows = []
pi_by_eps = {}
for eps in EPSILON_GRID:
    pi, info = sinkhorn_plan(C_eb_norm, epsilon=eps, return_info=True)
    pi_by_eps[float(eps)] = pi
    idx0, idx1 = sample_from_plan(pi, 4096, seed=100 + int(eps * 1000))
    diag = coupling_diagnostics(pi, C_eb_norm)
    eps_rows.append({
        "epsilon": float(eps),
        "expected_cost_normalized": float(diag["expected_cost"]),
        "cost_scale": float(C_eb_scale),
        "entropy": float(diag["entropy"]),
        "effective_support": float(diag["effective_support"]),
        "sinkhorn_converged": bool(info["sinkhorn_converged"]),
        "sinkhorn_n_iter": int(info["n_iter"]),
        "row_l1_error": float(info["row_l1_error"]),
        "col_l1_error": float(info["col_l1_error"]),
        "warning_message": info.get("warning_message", ""),
        "mean_pair_distance_pc20": float(np.linalg.norm(X1_eb[idx1] - X0_eb[idx0], axis=1).mean()),
        "mean_pair_distance_phate_display_only": float(np.linalg.norm(X1p_eb[idx1] - X0p_eb[idx0], axis=1).mean()),
        "cost_space": "standardized PC-20 median-normalized",
        "phate_distance_role": "display diagnostic only",
    })
eps_table = pd.DataFrame(eps_rows)
save_csv(OUT_DIR / "table4_A_sinkhorn_epsilon_ablation.csv", eps_table)
save_npz(CACHE_DIR / "exp2_epsilon_plans.npz", **{f"pi_eps_{str(k).replace('.', '_')}": v for k, v in pi_by_eps.items()})
eps_table


In [ ]:
# Epsilon pair visualization.
panels = []
for eps in [0.01, 0.05, 0.1, 0.5]:
    pi = pi_by_eps[float(eps)]
    idx0, idx1 = sample_from_plan(pi, 900, seed=120 + int(eps * 1000))
    panels.append({"title": f"epsilon={eps:g}", "idx0": idx0, "idx1": idx1, "color": PALETTE["ot"], "seed": 2})
plot_pair_panels(
    X0p_eb, X1p_eb, panels,
    "fig4_2b_epsilon_ablation_pairs.png",
    "Sinkhorn epsilon changes PC-20 coupling; displayed in PHATE only",
)


In [ ]:
minibatch_rows = []
mb_sizes = [64, 128, 256, 512, "full"]
rng = np.random.default_rng(202)
for mb in mb_sizes:
    repeats = 1 if mb == "full" else 10
    for rep in range(repeats):
        if mb == "full":
            x0_sub, x1_sub = X0_eb, X1_eb
        else:
            i0 = np.sort(rng.choice(len(X0_eb), size=min(int(mb), len(X0_eb)), replace=False))
            i1 = np.sort(rng.choice(len(X1_eb), size=min(int(mb), len(X1_eb)), replace=False))
            x0_sub, x1_sub = X0_eb[i0], X1_eb[i1]
        C_sub, scale_sub = compute_cost_matrix(x0_sub, x1_sub, normalize=True)
        pi_sub, info_sub = sinkhorn_plan(C_sub, epsilon=SINKHORN_EPSILON, return_info=True)
        diag = coupling_diagnostics(pi_sub, C_sub)
        minibatch_rows.append({
            "minibatch_size": str(mb),
            "repeat": int(rep),
            "n_source": int(len(x0_sub)),
            "n_target": int(len(x1_sub)),
            "expected_cost_normalized": float(diag["expected_cost"]),
            "effective_support": float(diag["effective_support"]),
            "entropy": float(diag["entropy"]),
            "sinkhorn_converged": bool(info_sub["sinkhorn_converged"]),
            "cost_scale": float(scale_sub),
        })
minibatch_raw = pd.DataFrame(minibatch_rows)
minibatch_summary = minibatch_raw.groupby("minibatch_size", sort=False).agg(
    repeats=("repeat", "count"),
    expected_cost_mean=("expected_cost_normalized", "mean"),
    expected_cost_std=("expected_cost_normalized", "std"),
    effective_support_mean=("effective_support", "mean"),
    effective_support_std=("effective_support", "std"),
).reset_index()
save_csv(OUT_DIR / "tableA_4_1_minibatch_ablation.csv", minibatch_summary)
save_csv(CACHE_DIR / "exp2_minibatch_ablation_raw.csv", minibatch_raw)
minibatch_summary


## Exp 3. Rectified Flow

Use the Exp 1 OT-CFM as the base model.  Reflow is trained against induced endpoints from model rollouts, but endpoint metrics are always computed against the real EB target distribution in standardized PC-20.


In [ ]:
def train_reflow_round(name: str, base_model, X0, steps: int = TRAINING_STEPS, seed: int = DEFAULT_SEED):
    endpoint_path = CACHE_DIR / f"{name}_induced_endpoint.npz"
    if endpoint_path.exists():
        Z = load_npz(endpoint_path)["endpoint"]
    else:
        Z = rollout_euler(base_model, X0, nfe=DEFAULT_NFE)
        save_npz(endpoint_path, endpoint=Z)
    pi_diag = np.eye(len(X0), len(Z), dtype=float)
    pi_diag /= pi_diag.sum()
    model, hist = train_or_load_model(name, X0, Z, pi_diag, steps=steps, seed=seed)
    return model, Z, hist

reflow1_model, Z1_1, reflow1_hist = train_reflow_round("exp3_reflow1", ot_model, X0_eb, seed=43)
reflow2_model, Z1_2, reflow2_hist = train_reflow_round("exp3_reflow2", reflow1_model, X0_eb, seed=44)


In [ ]:
models_exp3 = {
    "random_cfm": random_model,
    "ot_cfm": ot_model,
    "reflow_1": reflow1_model,
    "reflow_2": reflow2_model,
}
traj_exp3 = {}
rows = []
for name, model in models_exp3.items():
    z, traj, times = trajectory_rollout(model, X0_eb, nfe=DEFAULT_NFE, return_path=True)
    traj_exp3[name] = traj
    rows.append({
        "method": name,
        **evaluate_endpoint(z, X1_eb),
        "straightness_action_S": straightness_action_S(traj, times),
        "tortuosity_straightness": tortuosity_straightness(traj),
        "path_length": path_length(traj),
        "path_energy": path_energy(traj, times),
        "coarse_step_error_nfe4_vs_nfe64": coarse_step_error(model, X0_eb, nfe_coarse=4, nfe_fine=64),
        "endpoint_reference": "real EB target distribution in standardized PC-20",
    })
reflow_table = pd.DataFrame(rows)
save_csv(OUT_DIR / "table4_1_reflow_ablation.csv", reflow_table)

nfe_rows = []
for name, model in models_exp3.items():
    for nfe in NFE_GRID:
        z = rollout_euler(model, X0_eb, nfe=nfe)
        nfe_rows.append({"method": name, "nfe": int(nfe), **evaluate_endpoint(z, X1_eb)})
nfe_table = pd.DataFrame(nfe_rows)
save_csv(CACHE_DIR / "exp3_nfe_sensitivity.csv", nfe_table)
reflow_table


In [ ]:
plot_projected_trajectories(
    {"ot": traj_exp3["ot_cfm"], "reflow1": traj_exp3["reflow_1"], "reflow2": traj_exp3["reflow_2"]},
    X0p_eb, X1p_eb, pc_to_phate,
    "fig4_4_reflow_trajectories.png",
    "Reflow paths projected to PHATE; straightness is not biological validation",
)
plot_metric_lines(
    nfe_table, x="nfe", y="sliced_w2", hue="method",
    filename="fig4_4b_nfe_endpoint_error.png",
    title="NFE sensitivity against real EB target in PC-20",
)


## Exp 4. Coupling Diagnostic Table

For random CFM, OT-CFM, reflow-1, and reflow-2, compute endpoint, path geometry, off-manifold, and coarse-step diagnostics.  The off-manifold reference is all observed EB snapshots in standardized PC-20 when available.


In [ ]:
import gc

rows = []
exp4_progress_path = CACHE_DIR / "exp4_path_diagnostics_progress.json"
for name, model in models_exp3.items():
    save_json(exp4_progress_path, {"stage": "starting", "method": name, "completed_methods": [r["method"] for r in rows]})
    z, traj, times = trajectory_rollout(model, X0_eb, nfe=DEFAULT_NFE, return_path=True)
    row = {
        "method": name,
        **evaluate_endpoint(z, X1_eb),
        "path_length": path_length(traj),
        "path_energy_action": path_energy(traj, times),
        "straightness_action_S": straightness_action_S(traj, times),
        "tortuosity_straightness": tortuosity_straightness(traj),
        "off_manifold_knn_k15": off_manifold_knn(traj, off_manifold_reference_pc, k=15),
        "coarse_step_error_nfe4_vs_nfe64": coarse_step_error(model, X0_eb, nfe_coarse=4, nfe_fine=64),
        "off_manifold_reference": off_manifold_reference_note,
        "training_space": "standardized PC-20",
        "endpoint_metric_space": "standardized PC-20 real target",
    }
    rows.append(row)
    save_csv(OUT_DIR / "table4_1_path_geometry_diagnostics.csv", pd.DataFrame(rows))
    save_json(exp4_progress_path, {"stage": "completed", "method": name, "completed_methods": [r["method"] for r in rows]})
    del z, traj
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
path_diag_table = pd.DataFrame(rows)
save_csv(OUT_DIR / "table4_1_path_geometry_diagnostics.csv", path_diag_table)
plot_projected_trajectories(
    {"random": traj_exp3["random_cfm"], "ot": traj_exp3["ot_cfm"]},
    X0p_eb, X1p_eb, pc_to_phate,
    "fig4_5_random_vs_ot_projected_trajectories.png",
    "Random and OT trajectories projected to PHATE with local arrows",
)
path_diag_table


## Exp 5. Toy Branching: Coupling -> Branch Leakage

This controlled toy experiment uses toy fate labels only within toy data.  It trains random, OT, oracle fate-preserving, and label-shuffled CFM models, then evaluates terminal fate leakage with a classifier trained on observed terminal toy states.


In [ ]:
def load_toy_snapshots():
    df = pd.read_csv(TOY_CSV_PATH)
    df["time"] = df["time"].astype(float)
    return df

TOY_SOURCE_TIME = 0.5
TOY_TARGET_TIME = 1.0
TOY = load_toy_snapshots()
source_toy = TOY[np.isclose(TOY["time"], TOY_SOURCE_TIME)].reset_index(drop=True)
target_toy = TOY[np.isclose(TOY["time"], TOY_TARGET_TIME)].reset_index(drop=True)
if source_toy.empty or target_toy.empty:
    raise ValueError(f"Toy bridge {TOY_SOURCE_TIME}->{TOY_TARGET_TIME} is unavailable")
X0_toy_raw = source_toy[["state_1", "state_2"]].to_numpy(np.float32)
X1_toy_raw = target_toy[["state_1", "state_2"]].to_numpy(np.float32)
X0_toy, X1_toy, toy_state_meta = standardize_train_space(X0_toy_raw, X1_toy_raw)
y0_toy = source_toy["fate_label"].astype(str).to_numpy()
y1_toy = target_toy["fate_label"].astype(str).to_numpy()
y0_toy_eval = y0_toy.copy()
terminal_classifier = KNeighborsClassifier(n_neighbors=15, weights="distance")
terminal_classifier.fit(X1_toy, y1_toy)
print(TOY.groupby(["time", "fate_label"]).size().unstack(fill_value=0))
print("Exp 5 bridge:", TOY_SOURCE_TIME, "->", TOY_TARGET_TIME, "source labels", pd.Series(y0_toy).value_counts().to_dict(), "target labels", pd.Series(y1_toy).value_counts().to_dict())


In [ ]:
def fate_conditioned_plan(X0, X1, source_labels, target_labels, source_labels_for_plan=None):
    """Couple only within source-label to target-label blocks.

    The plan is used as a training pair sampler. It is row-balanced by source
    cells and only assigns mass to target cells with the same fate label when
    such targets exist.
    """
    source_labels = np.asarray(source_labels).astype(str)
    target_labels = np.asarray(target_labels).astype(str)
    labels_for_plan = source_labels if source_labels_for_plan is None else np.asarray(source_labels_for_plan).astype(str)
    pi = np.zeros((len(X0), len(X1)), dtype=float)
    for i, lab in enumerate(labels_for_plan):
        cols = np.flatnonzero(target_labels == lab)
        if cols.size == 0:
            cols = np.arange(len(X1))
        d2 = np.sum((X1[cols] - X0[i]) ** 2, axis=1)
        scale = max(float(np.median(d2[d2 > 0])) if np.any(d2 > 0) else 1.0, 1e-6)
        w = np.exp(-d2 / scale)
        w = w / np.clip(w.sum(), 1e-12, None)
        pi[i, cols] = w / len(X0)
    pi = pi / np.clip(pi.sum(), 1e-12, None)
    return pi

C_toy_norm, C_toy_scale = compute_cost_matrix(X0_toy, X1_toy, normalize=True)
pi_toy_random = independent_coupling(len(X0_toy), len(X1_toy))
pi_toy_ot, info_toy_ot = sinkhorn_plan(C_toy_norm, epsilon=SINKHORN_EPSILON, return_info=True)
pi_toy_oracle = fate_conditioned_plan(X0_toy, X1_toy, y0_toy, y1_toy)
y0_shuffled_for_plan = np.random.default_rng(123).permutation(y0_toy)
pi_toy_shuffled = fate_conditioned_plan(X0_toy, X1_toy, y0_toy, y1_toy, source_labels_for_plan=y0_shuffled_for_plan)

toy_plans = {
    "random": pi_toy_random,
    "ot": pi_toy_ot,
    "oracle_fate_preserving": pi_toy_oracle,
    "label_shuffled_control": pi_toy_shuffled,
}
save_npz(CACHE_DIR / "exp5_toy_plans.npz", **toy_plans)


In [ ]:
toy_models = {}
toy_rows = []
toy_pair_panels = []
for name, pi in toy_plans.items():
    model, hist = train_or_load_model(f"exp5_toy_{name}", X0_toy, X1_toy, pi, steps=TOY_TRAINING_STEPS, seed=60)
    toy_models[name] = model
    z, traj, times = trajectory_rollout(model, X0_toy, nfe=DEFAULT_NFE, return_path=True)
    pred_labels = terminal_classifier.predict(z)
    target_props = pd.Series(y1_toy).value_counts(normalize=True)
    pred_props = pd.Series(pred_labels).value_counts(normalize=True)
    branch_leakage = float(np.mean(pred_labels != y0_toy_eval))
    major_mask = y0_toy_eval == "major"
    rare_mask = y0_toy_eval == "rare"
    rare_err = abs(float(pred_props.get("rare", 0.0)) - float(target_props.get("rare", 0.0)))
    toy_rows.append({
        "method": name,
        "source_time": TOY_SOURCE_TIME,
        "target_time": TOY_TARGET_TIME,
        "branch_leakage": branch_leakage,
        "major_branch_leakage": float(np.mean(pred_labels[major_mask] != y0_toy_eval[major_mask])) if major_mask.any() else np.nan,
        "rare_branch_leakage": float(np.mean(pred_labels[rare_mask] != y0_toy_eval[rare_mask])) if rare_mask.any() else np.nan,
        "target_fate_mass_error": fate_mass_error(pred_labels, y1_toy),
        "rare_fate_mass_error": rare_err,
        "pred_rare_fraction": float(pred_props.get("rare", 0.0)),
        "target_rare_fraction": float(target_props.get("rare", 0.0)),
        "toy_label_scope": "toy-only controlled fate labels; not generalized to EB",
    })
    idx0, idx1 = sample_from_plan(pi, 600, seed=61)
    toy_pair_panels.append({"title": name.replace("_", " "), "idx0": idx0, "idx1": idx1, "source_labels": y0_toy[idx0], "seed": 61})

toy_diag_table = pd.DataFrame(toy_rows)
save_csv(OUT_DIR / "table4_2_toy_branch_diagnostics.csv", toy_diag_table)

# Toy pair figure in original toy coordinates, colored by source branch label.
fig, axes = plt.subplots(2, 2, figsize=(9, 8), squeeze=False)
label_color = {"major": PALETTE["major"], "rare": PALETTE["rare"], "progenitor": "0.45"}
for ax, panel in zip(axes.ravel(), toy_pair_panels):
    ax.scatter(X0_toy_raw[:, 0], X0_toy_raw[:, 1], s=8, color=PALETTE["source"], alpha=0.22, linewidths=0)
    ax.scatter(X1_toy_raw[:, 0], X1_toy_raw[:, 1], s=8, color=PALETTE["target"], alpha=0.18, linewidths=0)
    keep = sample_rows(len(panel["idx0"]), 100, seed=62)
    for k in keep:
        lab = str(panel["source_labels"][k])
        a, b = X0_toy_raw[panel["idx0"][k]], X1_toy_raw[panel["idx1"][k]]
        ax.plot([a[0], b[0]], [a[1], b[1]], color=label_color.get(lab, "0.35"), alpha=0.32, linewidth=0.8)
    ax.set_title(panel["title"])
    ax.set_xlabel("toy state 1")
    ax.set_ylabel("toy state 2")
save_figure(fig, "fig4_5b_toy_branching_pairs.png")
toy_diag_table


## Exp 6. Representation Space Changes Coupling

On the same toy cells, compare PCA-30 and 4D gene-program state spaces.  Expected costs are normalized and reported only within their own representation; raw expected cost is not ranked across representations.


In [ ]:
def load_toy_expression_representations():
    if ad is None:
        raise ImportError("anndata is required for toy pseudocount representation experiment")
    adata = ad.read_h5ad(TOY_H5AD_PATH)
    X_counts = adata.X.toarray() if hasattr(adata.X, "toarray") else np.asarray(adata.X)
    lib = np.maximum(X_counts.sum(axis=1, keepdims=True), 1.0)
    X_log = np.log1p(X_counts / lib * 1e4).astype(np.float32)
    pca_state = fit_pca_state_space(X_log, n_components=30, seed=42)
    programs = program_index_dict(adata, program_key="program", include_background=False)
    program_scores, program_names = readout_program_scores_from_matrix(X_log, programs)
    obs = adata.obs.reset_index(drop=True).copy()
    obs["time"] = obs["time"].astype(float)
    idx0 = np.flatnonzero(obs["time"].to_numpy() == obs["time"].min())
    idx1 = np.flatnonzero(obs["time"].to_numpy() == obs["time"].max())
    X0_pca_raw, X1_pca_raw = pca_state["coords"][idx0], pca_state["coords"][idx1]
    X0_prog_raw, X1_prog_raw = program_scores[idx0], program_scores[idx1]
    X0_pca, X1_pca, pca_std = standardize_train_space(X0_pca_raw, X1_pca_raw)
    X0_prog, X1_prog, prog_std = standardize_train_space(X0_prog_raw, X1_prog_raw)
    return {
        "adata": adata,
        "obs": obs,
        "X_log": X_log,
        "pca_state": pca_state,
        "programs": programs,
        "program_names": program_names,
        "program_scores": program_scores,
        "idx0": idx0,
        "idx1": idx1,
        "X0_pca": X0_pca,
        "X1_pca": X1_pca,
        "X0_prog": X0_prog,
        "X1_prog": X1_prog,
        "X0_prog_raw": X0_prog_raw,
        "X1_prog_raw": X1_prog_raw,
        "pca_std": pca_std,
        "prog_std": prog_std,
        "X0_viz": np.asarray(adata.obsm["X_toy_state"])[idx0].astype(np.float32),
        "X1_viz": np.asarray(adata.obsm["X_toy_state"])[idx1].astype(np.float32),
        "labels1": obs.iloc[idx1]["fate_label"].astype(str).to_numpy(),
    }

TOY_REP = load_toy_expression_representations()
C_pca, scale_pca = compute_cost_matrix(TOY_REP["X0_pca"], TOY_REP["X1_pca"], normalize=True)
C_prog, scale_prog = compute_cost_matrix(TOY_REP["X0_prog"], TOY_REP["X1_prog"], normalize=True)
pi_pca, info_pca = sinkhorn_plan(C_pca, epsilon=SINKHORN_EPSILON, return_info=True)
pi_prog, info_prog = sinkhorn_plan(C_prog, epsilon=SINKHORN_EPSILON, return_info=True)
rep_l1 = coupling_l1_distance(pi_pca, pi_prog)
rep_nn_overlap = coupling_topk_overlap(pi_pca, pi_prog, k=15)
rep_rows = []
for name, pi, C, scale, info in [
    ("pca30", pi_pca, C_pca, scale_pca, info_pca),
    ("program4", pi_prog, C_prog, scale_prog, info_prog),
]:
    diag = coupling_diagnostics(pi, C)
    rep_rows.append({
        "representation": name,
        "epsilon": SINKHORN_EPSILON,
        "coupling_l1_vs_other_representation": rep_l1,
        "topk_nn_overlap_k15": rep_nn_overlap,
        "effective_support": diag["effective_support"],
        "entropy": diag["entropy"],
        "expected_cost_normalized_within_representation": diag["expected_cost"],
        "raw_expected_cost_comparison_allowed": False,
        "sinkhorn_converged": info["sinkhorn_converged"],
    })
rep_coupling_table = pd.DataFrame(rep_rows)
save_csv(OUT_DIR / "table4_3_representation_coupling_diagnostics.csv", rep_coupling_table)
rep_coupling_table


In [ ]:
rep_models = {}
rep_metric_rows = []
terminal_fate_classifier = KNeighborsClassifier(n_neighbors=min(15, len(TOY_REP["X1_prog_raw"])), weights="distance")
terminal_fate_classifier.fit(TOY_REP["X1_prog_raw"], TOY_REP["labels1"])

for name, X0_train, X1_train, pi in [
    ("pca30", TOY_REP["X0_pca"], TOY_REP["X1_pca"], pi_pca),
    ("program4", TOY_REP["X0_prog"], TOY_REP["X1_prog"], pi_prog),
]:
    model, hist = train_or_load_model(f"exp6_toy_{name}", X0_train, X1_train, pi, steps=TOY_TRAINING_STEPS, seed=70)
    rep_models[name] = model
    z, traj, times = trajectory_rollout(model, X0_train, nfe=DEFAULT_NFE, return_path=True)

    native_endpoint_mmd = mmd_rbf(z, X1_train)
    native_sliced_w2 = sliced_w2(z, X1_train, seed=76)

    if name == "program4":
        pred_readout = z * TOY_REP["prog_std"]["std"] + TOY_REP["prog_std"]["mean"]
        fate_note = "terminal fate predicted from generated Program-4 readout with target-readout KNN classifier"
    else:
        z_raw = z * TOY_REP["pca_std"]["std"] + TOY_REP["pca_std"]["mean"]
        expr_pred = pca_inverse_transform(z_raw, TOY_REP["pca_state"])
        pred_readout, _ = readout_program_scores_from_matrix(expr_pred, TOY_REP["programs"])
        fate_note = "PCA endpoint inverse-transformed to expression, then read out as 4D gene-program scores for target-readout KNN classifier"

    readout_metrics = distribution_readout_metrics(pred_readout, TOY_REP["X1_prog_raw"])
    pred_fate = terminal_fate_classifier.predict(pred_readout)
    rep_metric_rows.append({
        "model_training_representation": name,
        "native_endpoint_mmd": native_endpoint_mmd,
        "native_sliced_w2": native_sliced_w2,
        "program_readout_mmd": readout_metrics["program_readout_mmd"],
        "program_readout_sliced_w2": readout_metrics["program_readout_sliced_w2"],
        "program_readout_mae": readout_metrics["program_readout_mean_abs_error"],
        "program_readout_mean_abs_error": readout_metrics["program_readout_mean_abs_error"],
        "fate_mass_error": fate_mass_error(pred_fate, TOY_REP["labels1"]),
        "fate_mass_error_note": fate_note,
        "native_metric_cross_representation_comparable": False,
        "native_metric_scope": f"{name} native representation only; do not rank native metric values across representations",
        "readout_space": "shared toy 4D gene-program scores",
        "claim_boundary": "toy representation sensitivity only; toy fate labels are not generalized to EB",
    })
state_space_metrics = pd.DataFrame(rep_metric_rows)
save_csv(OUT_DIR / "table4_4_state_space_model_metrics.csv", state_space_metrics)

# 2x2: PCA cost + PCA sampled pairs; program cost + program sampled pairs.
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, M, title in [(axes[0, 0], C_pca, "PCA-30 normalized cost"), (axes[1, 0], C_prog, "Program-4 normalized cost")]:
    rows = sample_rows(M.shape[0], min(120, M.shape[0]), seed=72)
    cols = sample_rows(M.shape[1], min(120, M.shape[1]), seed=73)
    im = ax.imshow(M[np.ix_(rows, cols)], aspect="auto", cmap="magma")
    ax.set_title(title)
    ax.set_xlabel("target subset")
    ax.set_ylabel("source subset")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
for ax, pi, title, color in [(axes[0, 1], pi_pca, "PCA-30 sampled pairs", PALETTE["ot"]), (axes[1, 1], pi_prog, "Program-4 sampled pairs", PALETTE["program"] )]:
    idx0, idx1 = sample_from_plan(pi, 700, seed=74)
    ax.scatter(TOY_REP["X0_viz"][:, 0], TOY_REP["X0_viz"][:, 1], s=8, color=PALETTE["source"], alpha=0.25, linewidths=0)
    ax.scatter(TOY_REP["X1_viz"][:, 0], TOY_REP["X1_viz"][:, 1], s=8, color=PALETTE["target"], alpha=0.20, linewidths=0)
    keep = sample_rows(len(idx0), 90, seed=75)
    for k in keep:
        a, b = TOY_REP["X0_viz"][idx0[k]], TOY_REP["X1_viz"][idx1[k]]
        ax.plot([a[0], b[0]], [a[1], b[1]], color=color, alpha=0.32, linewidth=0.8)
    ax.set_title(title)
    ax.set_xlabel("toy state 1")
    ax.set_ylabel("toy state 2")
fig.suptitle("Changing representation changes cost and sampled endpoint pairs", y=1.02)
save_figure(fig, "fig4_8_toy_representation_couplings.png")
state_space_metrics


## Exp 7. EB Representation Sensitivity: PC vs PHATE Coupling

This is a contrastive diagnostic only.  EB models are not trained in PHATE, and EB endpoint distributional metrics are not computed in PHATE.


In [ ]:
C_eb_pc, scale_eb_pc = compute_cost_matrix(X0_eb, X1_eb, normalize=True)
C_eb_phate, scale_eb_phate = compute_cost_matrix(X0p_eb, X1p_eb, normalize=True)
pi_eb_pc, info_eb_pc = sinkhorn_plan(C_eb_pc, epsilon=SINKHORN_EPSILON, return_info=True)
pi_eb_phate, info_eb_phate = sinkhorn_plan(C_eb_phate, epsilon=SINKHORN_EPSILON, return_info=True)
l1_eb = coupling_l1_distance(pi_eb_pc, pi_eb_phate)
overlap_eb = coupling_topk_overlap(pi_eb_pc, pi_eb_phate, k=15)
rows = []
for name, pi, C, info in [
    ("pc20_coupling", pi_eb_pc, C_eb_pc, info_eb_pc),
    ("phate2_contrastive_diagnostic", pi_eb_phate, C_eb_phate, info_eb_phate),
]:
    idx0, idx1 = sample_from_plan(pi, 4096, seed=82)
    diag = coupling_diagnostics(pi, C)
    rows.append({
        "representation": name,
        "epsilon": SINKHORN_EPSILON,
        "coupling_l1_vs_other": l1_eb,
        "topk_nn_overlap_k15": overlap_eb,
        "mean_sampled_pair_distance_pc20": float(np.linalg.norm(X1_eb[idx1] - X0_eb[idx0], axis=1).mean()),
        "mean_sampled_pair_distance_phate": float(np.linalg.norm(X1p_eb[idx1] - X0p_eb[idx0], axis=1).mean()),
        "normalized_expected_cost": diag["expected_cost"],
        "entropy": diag["entropy"],
        "effective_support": diag["effective_support"],
        "sinkhorn_converged": info["sinkhorn_converged"],
        "claim_boundary": "PHATE coupling is a contrastive display diagnostic only; no EB model trained in PHATE",
    })
eb_rep_table = pd.DataFrame(rows)
save_csv(OUT_DIR / "table4_5_eb_representation_coupling_diagnostics.csv", eb_rep_table)

idx_pc0, idx_pc1 = sample_from_plan(pi_eb_pc, 800, seed=84)
idx_ph0, idx_ph1 = sample_from_plan(pi_eb_phate, 800, seed=85)
plot_pair_panels(
    X0p_eb, X1p_eb,
    [
        {"title": "PC-20 coupling displayed in PHATE", "idx0": idx_pc0, "idx1": idx_pc1, "color": PALETTE["ot"], "seed": 84},
        {"title": "PHATE diagnostic coupling", "idx0": idx_ph0, "idx1": idx_ph1, "color": PALETTE["diagnostic"], "seed": 85},
    ],
    "fig4_8b_eb_pc_vs_phate_coupling.png",
    "EB representation sensitivity; PHATE is diagnostic only",
)
eb_rep_table


## Exp 8. Euclidean Chord vs Manifold-Aware Path

On toy data, compare straight endpoint interpolation with kNN-graph shortest paths on observed toy states.  Intermediate diagnostics use density percentile and kNN radius at `t = {0.25, 0.5, 0.75}`.


In [ ]:
from sklearn.neighbors import kneighbors_graph
from scipy.sparse.csgraph import connected_components, dijkstra

EXP8_T_VALUES = [0.25, 0.5, 0.75]
EXP8_GRAPH_K_GRID = [8, 12, 20, 30, 50, 80, 100, 150]
EXP8_CANDIDATE_PAIRS = 500 if SMOKE_MODE else 3000
EXP8_SELECTED_PAIRS = 8 if SMOKE_MODE else 16
EXP8_MIN_SELECTED_PAIRS = 6 if SMOKE_MODE else 10
EXP8_PATH_POINTS = 41


def build_exp8_knn_graph(points, k_grid=EXP8_GRAPH_K_GRID):
    """Build an undirected sklearn kNN graph and choose the first connected, not-overdense graph."""
    points = np.asarray(points, dtype=np.float32)
    graph_rows = []
    best = None
    for k in k_grid:
        kk = max(1, min(int(k), len(points) - 1))
        G = kneighbors_graph(points, n_neighbors=kk, mode="distance", include_self=False)
        G = G.maximum(G.T).tocsr()
        n_components, component_labels = connected_components(G, directed=False)
        density = float(G.nnz / max(G.shape[0] * G.shape[1], 1))
        row = {
            "k_graph": kk,
            "n_components": int(n_components),
            "n_edges_undirected": int(G.nnz // 2),
            "graph_density": density,
            "largest_component_fraction": float(np.bincount(component_labels).max() / len(points)),
        }
        graph_rows.append(row)
        if n_components == 1 and best is None:
            best = (G, row, graph_rows)
            if density <= 0.05:
                break
    if best is None:
        # Keep the least fragmented graph for explicit fallback diagnostics instead of silently using straight chords.
        best_idx = int(np.argmin([r["n_components"] for r in graph_rows]))
        kk = graph_rows[best_idx]["k_graph"]
        G = kneighbors_graph(points, n_neighbors=kk, mode="distance", include_self=False)
        G = G.maximum(G.T).tocsr()
        best = (G, graph_rows[best_idx], graph_rows)
    return best


def exp8_polyline_length(points):
    points = np.asarray(points, dtype=np.float32)
    if len(points) < 2:
        return 0.0
    return float(np.linalg.norm(np.diff(points, axis=0), axis=1).sum())


def exp8_point_on_polyline(points, t):
    points = np.asarray(points, dtype=np.float32)
    if len(points) == 0:
        raise ValueError("polyline must contain at least one point")
    if len(points) == 1:
        return points[0]
    seg = np.linalg.norm(np.diff(points, axis=0), axis=1)
    total = float(seg.sum())
    if total <= 1e-12:
        return points[min(int(round(float(t) * (len(points) - 1))), len(points) - 1)]
    arclen = np.r_[0.0, np.cumsum(seg) / total]
    return np.array([np.interp(float(t), arclen, points[:, d]) for d in range(points.shape[1])], dtype=np.float32)


def exp8_resample_polyline(points, n_points=EXP8_PATH_POINTS):
    grid = np.linspace(0.0, 1.0, int(n_points))
    return np.stack([exp8_point_on_polyline(points, t) for t in grid], axis=0)


def exp8_extract_path_indices(pred_row, source_node, target_node, dist_value, n_nodes):
    source_node = int(source_node)
    target_node = int(target_node)
    if source_node == target_node:
        return [source_node], False
    if not np.isfinite(dist_value):
        return [source_node, target_node], True
    path = [target_node]
    cur = target_node
    for _ in range(int(n_nodes) + 5):
        cur = int(pred_row[cur])
        if cur < 0:
            return [source_node, target_node], True
        path.append(cur)
        if cur == source_node:
            return path[::-1], False
    return [source_node, target_node], True


def exp8_density_scorer(reference_points, k=15):
    reference_points = np.asarray(reference_points, dtype=np.float32)
    kk = max(1, min(int(k), len(reference_points)))
    nn = NearestNeighbors(n_neighbors=kk).fit(reference_points)
    ref_radius = nn.kneighbors(reference_points, return_distance=True)[0][:, -1]
    ref_sorted = np.sort(ref_radius)

    def score(points):
        points = np.asarray(points, dtype=np.float32)
        if points.ndim == 1:
            points = points[None]
        radius = nn.kneighbors(points, return_distance=True)[0][:, -1]
        percentile = np.searchsorted(ref_sorted, radius, side="right") / len(ref_sorted) * 100.0
        return percentile.astype(float), radius.astype(float)

    return score, ref_radius


def exp8_candidate_diagnostics(source_points, target_points, pi, graph_points, graph, score_fn, n_candidates, seed=90):
    idx0, idx1 = sample_from_plan(pi, int(n_candidates), seed=seed)
    nearest_graph = NearestNeighbors(n_neighbors=1).fit(graph_points)
    source_nodes = nearest_graph.kneighbors(source_points[idx0], return_distance=False)[:, 0].astype(int)
    target_nodes = nearest_graph.kneighbors(target_points[idx1], return_distance=False)[:, 0].astype(int)
    unique_sources, inverse_sources = np.unique(source_nodes, return_inverse=True)
    dist_matrix, predecessors = dijkstra(graph, directed=False, indices=unique_sources, return_predecessors=True)
    if dist_matrix.ndim == 1:
        dist_matrix = dist_matrix[None, :]
        predecessors = predecessors[None, :]

    rows = []
    path_cache = {}
    for pair_id, (i, j, s_node, e_node, src_row) in enumerate(zip(idx0, idx1, source_nodes, target_nodes, inverse_sources)):
        start = np.asarray(source_points[int(i)], dtype=np.float32)
        end = np.asarray(target_points[int(j)], dtype=np.float32)
        dist_value = float(dist_matrix[int(src_row), int(e_node)])
        path_idx, used_fallback = exp8_extract_path_indices(
            predecessors[int(src_row)], int(s_node), int(e_node), dist_value, len(graph_points)
        )
        graph_path_nodes = np.asarray(graph_points[path_idx], dtype=np.float32)
        if used_fallback or len(graph_path_nodes) < 2:
            graph_path_nodes = np.vstack([start, end]).astype(np.float32)
        graph_length = exp8_polyline_length(graph_path_nodes)
        straight_length = float(np.linalg.norm(end - start))
        straight_mid = 0.5 * (start + end)
        graph_mid = exp8_point_on_polyline(graph_path_nodes, 0.5)
        spct, srad = score_fn(straight_mid)
        gpct, grad = score_fn(graph_mid)
        path_cache[int(pair_id)] = graph_path_nodes
        rows.append({
            "pair_id": int(pair_id),
            "source_idx": int(i),
            "target_idx": int(j),
            "source_graph_node": int(s_node),
            "target_graph_node": int(e_node),
            "euclidean_distance": straight_length,
            "graph_path_num_nodes": int(len(path_idx)),
            "graph_path_length": graph_length,
            "straight_path_length": straight_length,
            "path_length_ratio": graph_length / max(straight_length, 1e-12),
            "used_fallback": bool(used_fallback),
            "straight_mid_knn_radius": float(srad[0]),
            "graph_mid_knn_radius": float(grad[0]),
            "straight_minus_graph_mid_knn_radius": float(srad[0] - grad[0]),
            "straight_mid_density_percentile": float(spct[0]),
            "graph_mid_density_percentile": float(gpct[0]),
            "straight_minus_graph_density_percentile": float(spct[0] - gpct[0]),
        })
    return pd.DataFrame(rows), path_cache


def exp8_select_pairs(candidate_diag, max_pairs=EXP8_SELECTED_PAIRS, min_pairs=EXP8_MIN_SELECTED_PAIRS):
    diag = candidate_diag.copy()
    diag["selection_score"] = (
        diag["straight_minus_graph_density_percentile"].rank(pct=True)
        + diag["straight_minus_graph_mid_knn_radius"].rank(pct=True)
        + diag["euclidean_distance"].rank(pct=True)
        + diag["path_length_ratio"].rank(pct=True)
    )
    attempts = [
        (0.50, True, "nonfallback_nodes_ge4_upper_half_positive_midpoint_difference"),
        (0.35, True, "relaxed_distance_quantile_positive_midpoint_difference"),
        (0.00, True, "positive_midpoint_difference_any_distance"),
        (0.35, False, "relaxed_best_nonfallback_nodes_ge4"),
        (0.00, False, "best_nonfallback_nodes_ge4"),
    ]
    selected = pd.DataFrame()
    selected_reason = "no_pairs_selected"
    for q, require_positive, reason in attempts:
        distance_threshold = float(diag["euclidean_distance"].quantile(q))
        mask = (~diag["used_fallback"]) & (diag["graph_path_num_nodes"] >= 4) & (diag["euclidean_distance"] >= distance_threshold)
        if require_positive:
            mask &= (diag["straight_minus_graph_mid_knn_radius"] > 0) | (diag["straight_minus_graph_density_percentile"] > 0)
        pool = diag.loc[mask].sort_values("selection_score", ascending=False)
        if len(pool) >= min_pairs:
            selected = pool.head(max_pairs).copy()
            selected_reason = reason
            break
        if len(pool) > len(selected):
            selected = pool.copy()
            selected_reason = reason + "_too_few"
    if selected.empty:
        selected = diag.loc[~diag["used_fallback"]].sort_values("selection_score", ascending=False).head(max_pairs).copy()
        selected_reason = "last_resort_nonfallback_selection"
    selected["selected_reason"] = selected_reason
    selected["selected_rank"] = np.arange(len(selected), dtype=int)
    return selected


def exp8_stats_for_selected(selected_diag, source_points, target_points, path_cache, score_fn):
    stat_rows = []
    for _, row in selected_diag.iterrows():
        pair_id = int(row["pair_id"])
        source_idx = int(row["source_idx"])
        target_idx = int(row["target_idx"])
        start = np.asarray(source_points[source_idx], dtype=np.float32)
        end = np.asarray(target_points[target_idx], dtype=np.float32)
        graph_path_nodes = np.asarray(path_cache[pair_id], dtype=np.float32)
        for tval in EXP8_T_VALUES:
            straight_point = (1.0 - float(tval)) * start + float(tval) * end
            graph_point = exp8_point_on_polyline(graph_path_nodes, tval)
            spct, srad = score_fn(straight_point)
            gpct, grad = score_fn(graph_point)
            common = {
                "pair_id": pair_id,
                "source_idx": source_idx,
                "target_idx": target_idx,
                "graph_path_num_nodes": int(row["graph_path_num_nodes"]),
                "used_fallback": bool(row["used_fallback"]),
                "selected_reason": str(row["selected_reason"]),
                "t": float(tval),
            }
            stat_rows.append({**common, "path_type": "straight_chord", "density_radius_percentile": float(spct[0]), "knn_radius": float(srad[0])})
            stat_rows.append({**common, "path_type": "knn_graph_shortest_path", "density_radius_percentile": float(gpct[0]), "knn_radius": float(grad[0])})
    return pd.DataFrame(stat_rows)


def exp8_paired_differences(stats):
    wide = stats.pivot_table(index=["pair_id", "t"], columns="path_type", values=["density_radius_percentile", "knn_radius"])
    out = pd.DataFrame(index=wide.index)
    out["straight_minus_graph_density_percentile"] = wide[("density_radius_percentile", "straight_chord")] - wide[("density_radius_percentile", "knn_graph_shortest_path")]
    out["straight_minus_graph_knn_radius"] = wide[("knn_radius", "straight_chord")] - wide[("knn_radius", "knn_graph_shortest_path")]
    return out.reset_index()


def exp8_plot_supplement(stats, filename, title):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharex=True)
    path_order = ["straight_chord", "knn_graph_shortest_path"]
    colors = {"straight_chord": PALETTE["diagnostic"], "knn_graph_shortest_path": PALETTE["program"]}

    def grouped_boxplot(ax, metric, ylabel, panel_title):
        positions = []
        data = []
        facecolors = []
        tick_positions = []
        for ti, tval in enumerate(EXP8_T_VALUES):
            center = ti * 3.0
            tick_positions.append(center)
            vals_by_type = {}
            for offset, path_type in [(-0.42, path_order[0]), (0.42, path_order[1])]:
                sub = stats[(stats["t"] == tval) & (stats["path_type"] == path_type)].sort_values("pair_id")
                vals = sub[metric].to_numpy()
                vals_by_type[path_type] = sub[["pair_id", metric]].set_index("pair_id")[metric]
                positions.append(center + offset)
                data.append(vals)
                facecolors.append(colors[path_type])
            common_ids = vals_by_type[path_order[0]].index.intersection(vals_by_type[path_order[1]].index)
            for pid in common_ids:
                ax.plot(
                    [center - 0.42, center + 0.42],
                    [vals_by_type[path_order[0]].loc[pid], vals_by_type[path_order[1]].loc[pid]],
                    color="0.72", alpha=0.35, linewidth=0.7, zorder=1,
                )
        bp = ax.boxplot(data, positions=positions, widths=0.62, patch_artist=True, showfliers=False, zorder=2)
        for patch, color in zip(bp["boxes"], facecolors):
            patch.set_facecolor(color)
            patch.set_alpha(0.48)
        for median in bp["medians"]:
            median.set_color("0.12")
            median.set_linewidth(1.1)
        rng = np.random.default_rng(123)
        for pos, vals, color in zip(positions, data, facecolors):
            jitter = rng.normal(scale=0.045, size=len(vals))
            ax.scatter(np.full(len(vals), pos) + jitter, vals, s=10, color=color, alpha=0.45, linewidths=0, zorder=3)
        ax.set_xticks(tick_positions, [str(t) for t in EXP8_T_VALUES])
        ax.set_xlabel("intermediate time t")
        ax.set_ylabel(ylabel)
        ax.set_title(panel_title)
        ax.grid(axis="y", alpha=0.20)

    grouped_boxplot(axes[0], "density_radius_percentile", "density-radius percentile", "Panel A: density percentile")
    grouped_boxplot(axes[1], "knn_radius", "kNN radius", "Panel B: kNN radius")
    legend_handles = [
        plt.Line2D([0], [0], marker="s", linestyle="", color=colors[path_order[0]], label="straight chord", markersize=8, alpha=0.75),
        plt.Line2D([0], [0], marker="s", linestyle="", color=colors[path_order[1]], label="kNN graph path", markersize=8, alpha=0.75),
    ]
    axes[1].legend(handles=legend_handles, frameon=False, loc="best")
    fig.suptitle(title, y=1.02)
    return save_figure(fig, filename)


def exp8_plot_paths_2d(observed_points, source_points, target_points, selected_diag, path_cache, filename, title, max_pairs=16):
    fig, ax = plt.subplots(figsize=(6.4, 5.4))
    ax.scatter(observed_points[:, 0], observed_points[:, 1], s=6, color="0.72", alpha=0.24, linewidths=0, label="observed toy states")
    shown = selected_diag.sort_values("selected_rank").head(int(max_pairs))
    for n, (_, row) in enumerate(shown.iterrows()):
        pair_id = int(row["pair_id"])
        start = source_points[int(row["source_idx"])]
        end = target_points[int(row["target_idx"])]
        graph_path = exp8_resample_polyline(path_cache[pair_id], EXP8_PATH_POINTS)
        straight_path = np.linspace(start, end, EXP8_PATH_POINTS)
        show_label = n == 0
        ax.plot(straight_path[:, 0], straight_path[:, 1], color=PALETTE["diagnostic"], alpha=0.35, linewidth=1.0, linestyle="-", label="straight chord" if show_label else None)
        ax.plot(graph_path[:, 0], graph_path[:, 1], color=PALETTE["program"], alpha=0.70, linewidth=1.15, linestyle="--", label="kNN graph path" if show_label else None)
        for tval in EXP8_T_VALUES:
            spt = (1.0 - tval) * start + tval * end
            gpt = exp8_point_on_polyline(path_cache[pair_id], tval)
            ax.scatter(spt[0], spt[1], s=16, marker="o", facecolors="none", edgecolors=PALETTE["diagnostic"], linewidths=0.8, alpha=0.65)
            ax.scatter(gpt[0], gpt[1], s=18, marker="x", color=PALETTE["program"], linewidths=0.9, alpha=0.80)
    ax.set_title(title)
    ax.set_xlabel("toy state 1")
    ax.set_ylabel("toy state 2")
    ax.legend(frameon=False, loc="best")
    return save_figure(fig, filename)


# Toy Exp 8: build a real graph diagnostic, then select non-degenerate pairs.
all_toy_points = TOY[["state_1", "state_2"]].to_numpy(np.float32)
toy_graph, toy_graph_info, toy_graph_grid = build_exp8_knn_graph(all_toy_points)
toy_score_fn, toy_ref_radius = exp8_density_scorer(all_toy_points, k=15)
toy_pair_diag, toy_path_cache = exp8_candidate_diagnostics(
    X0_toy_raw,
    X1_toy_raw,
    pi_toy_ot,
    all_toy_points,
    toy_graph,
    toy_score_fn,
    n_candidates=EXP8_CANDIDATE_PAIRS,
    seed=90,
)
for key, value in toy_graph_info.items():
    toy_pair_diag[key] = value
toy_pair_diag["graph_k_grid_tried"] = ",".join(str(r["k_graph"]) for r in toy_graph_grid)
toy_pair_diag["graph_components_by_k"] = ";".join(f"{r['k_graph']}:{r['n_components']}" for r in toy_graph_grid)
toy_pair_diag["graph_selection_note"] = (
    f"selected k={toy_graph_info['k_graph']} with {toy_graph_info['n_components']} connected component(s); "
    "smaller grid entries are retained in graph_components_by_k"
)
toy_pair_diag["candidate_set_size"] = int(EXP8_CANDIDATE_PAIRS)
toy_selected = exp8_select_pairs(toy_pair_diag)
toy_pair_diag["selected_for_figure"] = toy_pair_diag["pair_id"].isin(toy_selected["pair_id"]).astype(bool)
toy_pair_diag = toy_pair_diag.merge(
    toy_selected[["pair_id", "selected_rank", "selected_reason"]],
    on="pair_id",
    how="left",
)
save_csv(OUT_DIR / "exp8_pair_selection_diagnostics.csv", toy_pair_diag)

exp8_stats = exp8_stats_for_selected(toy_selected, X0_toy_raw, X1_toy_raw, toy_path_cache, toy_score_fn)
save_csv(OUT_DIR / "exp8_off_manifold_stats.csv", exp8_stats)

exp8_plot_paths_2d(
    all_toy_points,
    X0_toy_raw,
    X1_toy_raw,
    toy_selected,
    toy_path_cache,
    "fig4_10_chord_vs_manifold_path.png",
    "Euclidean chord versus density-aware kNN graph diagnostic paths",
    max_pairs=min(16, len(toy_selected)),
)
exp8_plot_supplement(
    exp8_stats,
    "fig4_10_supp_off_manifold_statistics.png",
    "Toy off-manifold diagnostics; graph paths are density-aware diagnostics, not true lineage",
)

toy_diffs = exp8_paired_differences(exp8_stats)
toy_diff_summary = toy_diffs.groupby("t")[["straight_minus_graph_density_percentile", "straight_minus_graph_knn_radius"]].mean()
toy_selected_nonfallback = int((~toy_selected["used_fallback"].astype(bool)).sum())
toy_selected_nodes_ge4 = int((toy_selected["graph_path_num_nodes"] >= 4).sum())
toy_candidate_fallback_count = int(toy_pair_diag["used_fallback"].astype(bool).sum())
toy_stats_not_identical = bool(
    (toy_diffs["straight_minus_graph_density_percentile"].abs().max() > 1e-9)
    or (toy_diffs["straight_minus_graph_knn_radius"].abs().max() > 1e-9)
)
toy_exp8_usable = bool(
    len(toy_selected) >= EXP8_MIN_SELECTED_PAIRS
    and toy_selected_nonfallback == len(toy_selected)
    and toy_selected_nodes_ge4 >= max(1, int(0.9 * len(toy_selected)))
    and toy_stats_not_identical
    and (
        toy_diff_summary["straight_minus_graph_knn_radius"].max() > 0
        or toy_diff_summary["straight_minus_graph_density_percentile"].max() > 0
    )
)
print("Exp 8 graph construction grid:")
display(pd.DataFrame(toy_graph_grid))
print("Exp 8 selected pairs:", len(toy_selected))
print("Exp 8 fallback candidate pairs:", toy_candidate_fallback_count)
print("Exp 8 selected nonfallback pairs:", toy_selected_nonfallback)
print("Exp 8 selected graph_path_num_nodes>=4:", toy_selected_nodes_ge4)
print("Exp 8 toy usable:", toy_exp8_usable)
print("Mean straight-minus-graph diagnostics by t:")
display(toy_diff_summary)


# EB real-data manifold diagnostic is implemented in the following Exp 8b cell.
# This toy cell does not generate EB artifacts, preventing fallback-based or metric-filtered EB examples.
eb_variant_generated = False
print("Dedicated EB Exp 8b cell handles EB PC-20 graph diagnostic; no EB fallback generated from toy cell.")

exp8_stats.groupby(["path_type", "t"])[["density_radius_percentile", "knn_radius"]].mean()

### Exp 8b. EB 20D PC Real-Data Manifold Diagnostic

This EB variant is a density-aware graph diagnostic in the standardized PC-20 cost/state space.  PHATE is used only to display selected paths.  Endpoint pairs are a deterministic seeded sample from the PC-20 OT coupling; main-figure pairs are evenly spaced among connected sampled pairs and are not selected by off-manifold score.  The graph path is not interpreted as true lineage.

In [ ]:
from sklearn.neighbors import kneighbors_graph
from scipy.sparse.csgraph import connected_components, dijkstra

EXP8_EB_SEED = 192
EXP8_EB_N_PAIRS_ALL = 200 if SMOKE_MODE else 500
EXP8_EB_N_MAIN_FIGURE_PAIRS = 6 if SMOKE_MODE else 10
EXP8_EB_K_GRID = [10, 15, 20, 30, 50, 80, 100]
EXP8_EB_T_VALUES = [0.25, 0.5, 0.75]
EXP8_EB_PATH_POINTS = 41
EXP8_EB_PAIR_SELECTION_MODE = "ot_coupling_seeded_sample_no_metric_filter"
EXP8_EB_MAIN_SELECTION_MODE = "evenly_spaced_connected_pair_ids_no_metric_filter"


def exp8_eb_polyline_length(points):
    points = np.asarray(points, dtype=np.float32)
    if len(points) < 2:
        return 0.0
    return float(np.linalg.norm(np.diff(points, axis=0), axis=1).sum())


def exp8_eb_point_on_polyline(points, t):
    points = np.asarray(points, dtype=np.float32)
    if len(points) == 0:
        raise ValueError("polyline must contain at least one point")
    if len(points) == 1:
        return points[0]
    seg = np.linalg.norm(np.diff(points, axis=0), axis=1)
    total = float(seg.sum())
    if total <= 1e-12:
        return points[min(int(round(float(t) * (len(points) - 1))), len(points) - 1)]
    arclen = np.r_[0.0, np.cumsum(seg) / total]
    return np.array([np.interp(float(t), arclen, points[:, d]) for d in range(points.shape[1])], dtype=np.float32)


def exp8_eb_resample_polyline(points, n_points=EXP8_EB_PATH_POINTS):
    return np.stack([exp8_eb_point_on_polyline(points, t) for t in np.linspace(0.0, 1.0, int(n_points))], axis=0)


def exp8_eb_extract_path_indices(pred_row, source_node, target_node, dist_value, n_nodes):
    source_node = int(source_node)
    target_node = int(target_node)
    if source_node == target_node:
        return [source_node], False
    if not np.isfinite(dist_value):
        return [source_node, target_node], True
    path = [target_node]
    cur = target_node
    for _ in range(int(n_nodes) + 5):
        cur = int(pred_row[cur])
        if cur < 0:
            return [source_node, target_node], True
        path.append(cur)
        if cur == source_node:
            return path[::-1], False
    return [source_node, target_node], True


def exp8_eb_density_scorer(reference_pc20, k=15):
    reference_pc20 = np.asarray(reference_pc20, dtype=np.float32)
    kk = max(1, min(int(k), len(reference_pc20)))
    nn = NearestNeighbors(n_neighbors=kk).fit(reference_pc20)
    ref_radius = nn.kneighbors(reference_pc20, return_distance=True)[0][:, -1]
    ref_sorted = np.sort(ref_radius)

    def score(points):
        points = np.asarray(points, dtype=np.float32)
        if points.ndim == 1:
            points = points[None]
        radius = nn.kneighbors(points, return_distance=True)[0][:, -1]
        percentile = np.searchsorted(ref_sorted, radius, side="right") / len(ref_sorted) * 100.0
        return percentile.astype(float), radius.astype(float)

    return score


def exp8_eb_load_or_compute_ot_plan():
    coupling_path = CACHE_DIR / "exp1_eb_couplings.npz"
    if coupling_path.exists():
        z = np.load(coupling_path, allow_pickle=True)
        if "pi_ot" in z.files and z["pi_ot"].shape == (len(X0_eb), len(X1_eb)):
            return np.asarray(z["pi_ot"], dtype=float), "loaded_exp1_pc20_ot_coupling"
    C_tmp, _ = compute_cost_matrix(X0_eb, X1_eb, normalize=True)
    pi_tmp, _ = sinkhorn_plan(C_tmp, epsilon=SINKHORN_EPSILON, return_info=True)
    return np.asarray(pi_tmp, dtype=float), "computed_temporary_pc20_ot_coupling"


def exp8_eb_build_graph_grid(reference_pc20, source_nodes, target_nodes, k_grid=EXP8_EB_K_GRID):
    reference_pc20 = np.asarray(reference_pc20, dtype=np.float32)
    graph_rows = []
    graphs = {}
    labels_by_k = {}
    for k in k_grid:
        kk = max(1, min(int(k), len(reference_pc20) - 1))
        G = kneighbors_graph(reference_pc20, n_neighbors=kk, mode="distance", include_self=False)
        G = G.maximum(G.T).tocsr()
        n_components, component_labels = connected_components(G, directed=False)
        connected = component_labels[source_nodes] == component_labels[target_nodes]
        density = float(G.nnz / max(G.shape[0] * G.shape[1], 1))
        row = {
            "k_graph": kk,
            "n_components": int(n_components),
            "largest_component_fraction": float(np.bincount(component_labels).max() / len(reference_pc20)),
            "graph_density": density,
            "n_edges_undirected": int(G.nnz // 2),
            "endpoint_connected_fraction": float(connected.mean()) if len(connected) else 0.0,
        }
        graph_rows.append(row)
        graphs[kk] = G
        labels_by_k[kk] = component_labels
    connected_candidates = [r for r in graph_rows if r["endpoint_connected_fraction"] >= 0.80]
    if connected_candidates:
        selected_info = sorted(connected_candidates, key=lambda r: (r["k_graph"], r["graph_density"]))[0]
    else:
        selected_info = sorted(graph_rows, key=lambda r: (-r["endpoint_connected_fraction"], r["k_graph"]))[0]
    selected_k = selected_info["k_graph"]
    return graphs[selected_k], selected_info, graph_rows, labels_by_k[selected_k]


def exp8_eb_save_figure_pair(fig, png_name, pdf_name):
    for name in [png_name, pdf_name]:
        path = FIG_DIR / name
        path.parent.mkdir(parents=True, exist_ok=True)
        fig.tight_layout()
        fig.savefig(path, bbox_inches="tight", dpi=220 if name.endswith(".png") else None)
    plt.close(fig)


def exp8_eb_plot_statistics(stats, filename_png, filename_pdf):
    connected = stats[~stats["used_fallback"].astype(bool)].copy()
    fig, axes = plt.subplots(1, 2, figsize=(11.4, 4.2), sharex=True)
    path_order = ["straight_chord", "knn_graph_shortest_path"]
    colors = {"straight_chord": PALETTE["diagnostic"], "knn_graph_shortest_path": PALETTE["program"]}

    def grouped_boxplot(ax, metric, ylabel, title):
        positions, data, facecolors, tick_positions = [], [], [], []
        for ti, tval in enumerate(EXP8_EB_T_VALUES):
            center = ti * 3.0
            tick_positions.append(center)
            vals_by_type = {}
            for offset, path_type in [(-0.42, path_order[0]), (0.42, path_order[1])]:
                sub = connected[(connected["t"] == tval) & (connected["path_type"] == path_type)].sort_values("pair_id")
                vals = sub[metric].to_numpy()
                vals_by_type[path_type] = sub[["pair_id", metric]].set_index("pair_id")[metric]
                positions.append(center + offset)
                data.append(vals)
                facecolors.append(colors[path_type])
            common_ids = vals_by_type[path_order[0]].index.intersection(vals_by_type[path_order[1]].index)
            for pid in common_ids:
                ax.plot(
                    [center - 0.42, center + 0.42],
                    [vals_by_type[path_order[0]].loc[pid], vals_by_type[path_order[1]].loc[pid]],
                    color="0.70", alpha=0.22, linewidth=0.55, zorder=1,
                )
        bp = ax.boxplot(data, positions=positions, widths=0.62, patch_artist=True, showfliers=False, zorder=2)
        for patch, color in zip(bp["boxes"], facecolors):
            patch.set_facecolor(color)
            patch.set_alpha(0.46)
        for median in bp["medians"]:
            median.set_color("0.12")
            median.set_linewidth(1.1)
        rng = np.random.default_rng(193)
        for pos, vals, color in zip(positions, data, facecolors):
            if len(vals) == 0:
                continue
            sample = vals if len(vals) <= 180 else rng.choice(vals, size=180, replace=False)
            jitter = rng.normal(scale=0.045, size=len(sample))
            ax.scatter(np.full(len(sample), pos) + jitter, sample, s=8, color=color, alpha=0.32, linewidths=0, zorder=3)
        ax.set_xticks(tick_positions, [str(t) for t in EXP8_EB_T_VALUES])
        ax.set_xlabel("intermediate time t")
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.grid(axis="y", alpha=0.20)

    grouped_boxplot(axes[0], "density_radius_percentile", "kNN-radius percentile", "Panel A: higher = farther from EB support")
    grouped_boxplot(axes[1], "knn_radius", "20D PC kNN radius", "Panel B: local support distance")
    handles = [
        plt.Line2D([0], [0], marker="s", linestyle="", color=colors[path_order[0]], label="straight chord", markersize=8, alpha=0.75),
        plt.Line2D([0], [0], marker="s", linestyle="", color=colors[path_order[1]], label="20D kNN graph path", markersize=8, alpha=0.75),
    ]
    axes[1].legend(handles=handles, frameon=False, loc="best")
    fig.suptitle("EB 20D off-manifold diagnostic; PHATE is not used for metrics", y=1.02)
    exp8_eb_save_figure_pair(fig, filename_png, filename_pdf)


# EB data and endpoint pairs. Metrics and graph construction use standardized PC-20 only.
X_eb_ref_pc20 = np.asarray(EB["pcs20_all"], dtype=np.float32)
X0_eb = np.asarray(EB["X0_pc"], dtype=np.float32)
X1_eb = np.asarray(EB["X1_pc"], dtype=np.float32)
pi_eb_for_exp8, eb_pair_plan_source = exp8_eb_load_or_compute_ot_plan()
idx0_eb_pairs, idx1_eb_pairs = sample_from_plan(pi_eb_for_exp8, EXP8_EB_N_PAIRS_ALL, seed=EXP8_EB_SEED)
source_global_nodes = EB["idx_source"][idx0_eb_pairs].astype(int)
target_global_nodes = EB["idx_target"][idx1_eb_pairs].astype(int)

G_eb, eb_graph_info, eb_graph_grid, eb_component_labels = exp8_eb_build_graph_grid(
    X_eb_ref_pc20, source_global_nodes, target_global_nodes, k_grid=EXP8_EB_K_GRID
)
eb_score_fn = exp8_eb_density_scorer(X_eb_ref_pc20, k=15)

unique_sources, inverse_sources = np.unique(source_global_nodes, return_inverse=True)
dist_matrix, predecessors = dijkstra(G_eb, directed=False, indices=unique_sources, return_predecessors=True)
if dist_matrix.ndim == 1:
    dist_matrix = dist_matrix[None, :]
    predecessors = predecessors[None, :]

path_cache = {}
diag_rows = []
stat_rows = []
for pair_id, (i0, i1, s_node, t_node, src_row) in enumerate(zip(idx0_eb_pairs, idx1_eb_pairs, source_global_nodes, target_global_nodes, inverse_sources)):
    start = X0_eb[int(i0)]
    end = X1_eb[int(i1)]
    dist_value = float(dist_matrix[int(src_row), int(t_node)])
    path_idx, used_fallback = exp8_eb_extract_path_indices(
        predecessors[int(src_row)], int(s_node), int(t_node), dist_value, len(X_eb_ref_pc20)
    )
    graph_path_nodes = X_eb_ref_pc20[path_idx]
    if used_fallback or len(graph_path_nodes) < 2:
        graph_path_nodes = np.vstack([start, end]).astype(np.float32)
    path_cache[int(pair_id)] = graph_path_nodes
    straight_len = float(np.linalg.norm(end - start))
    graph_len = exp8_eb_polyline_length(graph_path_nodes)
    connected = bool((not used_fallback) and len(path_idx) >= 2)
    diag_rows.append({
        "pair_id": int(pair_id),
        "source_idx": int(i0),
        "target_idx": int(i1),
        "source_global_node": int(s_node),
        "target_global_node": int(t_node),
        "source_timepoint": str(EB["source_time"]),
        "target_timepoint": str(EB["target_time"]),
        "euclidean_distance_pc20": straight_len,
        "graph_path_num_nodes": int(len(path_idx)),
        "graph_path_length_pc20": graph_len,
        "straight_path_length_pc20": straight_len,
        "path_length_ratio": graph_len / max(straight_len, 1e-12),
        "connected_pair": connected,
        "used_fallback": bool(used_fallback),
        "pair_selection_mode": EXP8_EB_PAIR_SELECTION_MODE,
        "main_figure_pair_selection_mode": EXP8_EB_MAIN_SELECTION_MODE,
        "no_metric_based_pair_selection": True,
        "state_space": "20D PC X_cost",
        "visualization_space": "PHATE 2D only",
        "selected_k_graph": int(eb_graph_info["k_graph"]),
        "graph_density": float(eb_graph_info["graph_density"]),
        "graph_components_by_k": ";".join(f"{r['k_graph']}:{r['n_components']}" for r in eb_graph_grid),
        "endpoint_connected_fraction_by_k": ";".join(f"{r['k_graph']}:{r['endpoint_connected_fraction']:.3f}" for r in eb_graph_grid),
        "interpretation": "density-aware diagnostic, not lineage inference",
    })
    for tval in EXP8_EB_T_VALUES:
        straight_point = (1.0 - float(tval)) * start + float(tval) * end
        graph_point = exp8_eb_point_on_polyline(graph_path_nodes, tval)
        spct, srad = eb_score_fn(straight_point)
        gpct, grad = eb_score_fn(graph_point)
        diff_pct = float(spct[0] - gpct[0])
        diff_rad = float(srad[0] - grad[0])
        common = {
            "pair_id": int(pair_id),
            "source_idx": int(i0),
            "target_idx": int(i1),
            "source_global_node": int(s_node),
            "target_global_node": int(t_node),
            "t": float(tval),
            "graph_path_num_nodes": int(len(path_idx)),
            "connected_pair": connected,
            "used_fallback": bool(used_fallback),
            "pair_selection_mode": EXP8_EB_PAIR_SELECTION_MODE,
            "main_figure_pair_selection_mode": EXP8_EB_MAIN_SELECTION_MODE,
            "no_metric_based_pair_selection": True,
            "state_space": "20D PC X_cost",
            "visualization_space": "PHATE 2D only",
            "straight_minus_graph_density_percentile": diff_pct,
            "straight_minus_graph_knn_radius": diff_rad,
            "interpretation": "density-aware diagnostic, not lineage inference",
        }
        stat_rows.append({**common, "path_type": "straight_chord", "density_radius_percentile": float(spct[0]), "knn_radius": float(srad[0])})
        stat_rows.append({**common, "path_type": "knn_graph_shortest_path", "density_radius_percentile": float(gpct[0]), "knn_radius": float(grad[0])})

eb_pair_diag = pd.DataFrame(diag_rows)
connected_pair_ids = eb_pair_diag.loc[eb_pair_diag["connected_pair"].astype(bool), "pair_id"].to_numpy(dtype=int)
if connected_pair_ids.size:
    n_main = min(EXP8_EB_N_MAIN_FIGURE_PAIRS, connected_pair_ids.size)
    main_positions = np.linspace(0, connected_pair_ids.size - 1, n_main).round().astype(int)
    main_pair_ids = np.unique(connected_pair_ids[main_positions])
else:
    main_pair_ids = np.array([], dtype=int)
eb_pair_diag["selected_for_main_figure"] = eb_pair_diag["pair_id"].isin(main_pair_ids)
eb_pair_diag["main_figure_selected_rank"] = np.nan
for rank, pid in enumerate(main_pair_ids):
    eb_pair_diag.loc[eb_pair_diag["pair_id"] == int(pid), "main_figure_selected_rank"] = int(rank)
save_csv(OUT_DIR / "exp8_eb_pair_selection_diagnostics.csv", eb_pair_diag)

eb_stats = pd.DataFrame(stat_rows)
eb_stats["selected_for_main_figure"] = eb_stats["pair_id"].isin(main_pair_ids)
save_csv(OUT_DIR / "exp8_eb_off_manifold_stats.csv", eb_stats)

connected_stats = eb_stats[~eb_stats["used_fallback"].astype(bool)].copy()
connected_diff = connected_stats.drop_duplicates(["pair_id", "t"])[[
    "pair_id", "t", "straight_minus_graph_density_percentile", "straight_minus_graph_knn_radius"
]].copy()
positive_fraction = connected_diff.groupby("t").apply(
    lambda g: pd.Series({
        "density_percentile": float((g["straight_minus_graph_density_percentile"] > 0).mean()),
        "knn_radius": float((g["straight_minus_graph_knn_radius"] > 0).mean()),
    })
).to_dict(orient="index")
mean_median = connected_diff.groupby("t").agg(
    mean_straight_minus_graph_density_percentile=("straight_minus_graph_density_percentile", "mean"),
    median_straight_minus_graph_density_percentile=("straight_minus_graph_density_percentile", "median"),
    mean_straight_minus_graph_knn_radius=("straight_minus_graph_knn_radius", "mean"),
    median_straight_minus_graph_knn_radius=("straight_minus_graph_knn_radius", "median"),
).to_dict(orient="index")
connected_fraction = float(eb_pair_diag["connected_pair"].mean()) if len(eb_pair_diag) else 0.0
fallback_fraction = float(eb_pair_diag["used_fallback"].mean()) if len(eb_pair_diag) else 0.0
shortcut_risk_note = (
    "low graph density for selected k; shortest paths are still diagnostic and may shortcut manifold geometry"
    if eb_graph_info["graph_density"] <= 0.01
    else "selected graph is relatively dense; shortcut risk is elevated, interpret as diagnostic only"
)
support_density = {
    str(t): bool(positive_fraction.get(t, {}).get("knn_radius", 0.0) >= 0.5 or positive_fraction.get(t, {}).get("density_percentile", 0.0) >= 0.5)
    for t in EXP8_EB_T_VALUES
}
summary = {
    "seed": EXP8_EB_SEED,
    "dataset": "EB",
    "state_space": "20D PC X_cost",
    "visualization_space": "PHATE 2D only",
    "source_timepoint": str(EB["source_time"]),
    "target_timepoint": str(EB["target_time"]),
    "n_pooled_cells": int(len(X_eb_ref_pc20)),
    "pc_dimension": int(X_eb_ref_pc20.shape[1]),
    "n_source_cells": int(len(X0_eb)),
    "n_target_cells": int(len(X1_eb)),
    "n_pairs_all": int(len(eb_pair_diag)),
    "n_pairs_connected": int(eb_pair_diag["connected_pair"].sum()),
    "connected_pair_fraction": connected_fraction,
    "n_main_figure_pairs": int(len(main_pair_ids)),
    "pair_selection_mode": EXP8_EB_PAIR_SELECTION_MODE,
    "main_figure_pair_selection_mode": EXP8_EB_MAIN_SELECTION_MODE,
    "pair_plan_source": eb_pair_plan_source,
    "selected_k_graph": int(eb_graph_info["k_graph"]),
    "graph_components_by_k": {str(r["k_graph"]): int(r["n_components"]) for r in eb_graph_grid},
    "endpoint_connected_fraction_by_k": {str(r["k_graph"]): float(r["endpoint_connected_fraction"]) for r in eb_graph_grid},
    "graph_density": float(eb_graph_info["graph_density"]),
    "largest_component_fraction": float(eb_graph_info["largest_component_fraction"]),
    "shortcut_risk_note": shortcut_risk_note,
    "fallback_pair_fraction": fallback_fraction,
    "positive_fraction_all_pairs_by_t": {str(k): v for k, v in positive_fraction.items()},
    "mean_median_straight_minus_graph_by_t": {str(k): v for k, v in mean_median.items()},
    "support_by_t_positive_fraction_ge_0_5": support_density,
    "no_metric_based_pair_selection": True,
    "interpretation": "density-aware diagnostic, not lineage inference",
}
save_json(OUT_DIR / "exp8_eb_summary.json", summary)

# PHATE display only. Define the PC->PHATE display mapper if the previous display cell was not executed.
X_eb_ref_phate = np.asarray(EB["phate_all"], dtype=np.float32)
if "pc_to_phate" not in globals():
    pc_to_phate_knn = KNeighborsClassifier(n_neighbors=min(15, len(EB["pcs20_all"])), weights="distance")
    pc_to_phate_knn.fit(EB["pcs20_all"], np.arange(len(EB["pcs20_all"])))

    def pc_to_phate(points_pc):
        points_pc = np.asarray(points_pc, dtype=np.float32)
        dist, ind = pc_to_phate_knn.kneighbors(points_pc, return_distance=True)
        w = 1.0 / np.clip(dist, 1e-6, None)
        w = w / w.sum(axis=1, keepdims=True)
        return np.einsum("nk,nkd->nd", w, EB["phate_all"][ind])

fig, ax = plt.subplots(figsize=(6.8, 5.6))
ax.scatter(X_eb_ref_phate[:, 0], X_eb_ref_phate[:, 1], s=4, color="0.70", alpha=0.13, linewidths=0)
for n, pid in enumerate(main_pair_ids):
    row = eb_pair_diag.loc[eb_pair_diag["pair_id"] == int(pid)].iloc[0]
    start = X0_eb[int(row["source_idx"])]
    end = X1_eb[int(row["target_idx"])]
    straight_pc = np.linspace(start, end, EXP8_EB_PATH_POINTS).astype(np.float32)
    graph_pc = exp8_eb_resample_polyline(path_cache[int(pid)], EXP8_EB_PATH_POINTS)
    straight_phate = pc_to_phate(straight_pc)
    graph_phate = pc_to_phate(graph_pc)
    label_once = n == 0
    ax.plot(straight_phate[:, 0], straight_phate[:, 1], color=PALETTE["diagnostic"], linestyle="-", linewidth=1.0, alpha=0.46, label="straight PC-20 chord" if label_once else None)
    ax.plot(graph_phate[:, 0], graph_phate[:, 1], color=PALETTE["program"], linestyle="--", linewidth=1.15, alpha=0.78, label="20D kNN graph path" if label_once else None)
    for tval in EXP8_EB_T_VALUES:
        spt = pc_to_phate(((1.0 - tval) * start + tval * end)[None])[0]
        gpt = pc_to_phate(exp8_eb_point_on_polyline(path_cache[int(pid)], tval)[None])[0]
        ax.scatter(spt[0], spt[1], s=18, marker="o", facecolors="none", edgecolors=PALETTE["diagnostic"], linewidths=0.85, alpha=0.70)
        ax.scatter(gpt[0], gpt[1], s=22, marker="x", color=PALETTE["program"], linewidths=0.95, alpha=0.88)
ax.set_title("EB 20D graph diagnostic paths; PHATE display only, not lineage ground truth")
ax.set_xlabel("PHATE 1 (display only)")
ax.set_ylabel("PHATE 2 (display only)")
ax.legend(frameon=False, loc="best")
exp8_eb_save_figure_pair(fig, "fig4_10_eb_chord_vs_graph_path_phate.png", "fig4_10_eb_chord_vs_graph_path_phate.pdf")

exp8_eb_plot_statistics(
    eb_stats,
    "fig4_10_eb_off_manifold_statistics.png",
    "fig4_10_eb_off_manifold_statistics.pdf",
)

print("EB Exp 8 graph grid:")
display(pd.DataFrame(eb_graph_grid))
print("EB Exp 8 summary:")
print(json.dumps(json_ready(summary), indent=2, sort_keys=True))
connected_diff.groupby("t")[["straight_minus_graph_density_percentile", "straight_minus_graph_knn_radius"]].agg(["mean", "median"])

## Exp 9. EB Equal-Depth Subsampling

Use all EB time points.  If no observed state labels are present, create coarse state bins with k-means in standardized PC-20.  Counts are reported as sampling-depth diagnostics and normalized proportions, not absolute biological abundance.


In [ ]:
labels_all = EB["labels"]
pcs_all = EB["pcs20_all"]
unique_times = sorted(np.unique(labels_all).tolist(), key=lambda s: int(s) if str(s).isdigit() else str(s))
# EB npz has no cell-type/state labels; create coarse bins on PC-20.
n_bins = min(8, max(2, int(np.sqrt(len(unique_times) * 8))))
kmeans = KMeans(n_clusters=n_bins, random_state=42, n_init=10)
state_bins = kmeans.fit_predict(pcs_all).astype(str)
all_bins = sorted(np.unique(state_bins).tolist(), key=lambda s: int(s) if str(s).isdigit() else s)

counts_rows = []
for t in unique_times:
    mask = labels_all == t
    total = int(mask.sum())
    count_by_bin = pd.Series(state_bins[mask]).value_counts()
    for b in all_bins:
        c = int(count_by_bin.get(b, 0))
        counts_rows.append({"time": t, "state_bin": b, "n_cells": c, "total_time_cells": total, "proportion": float(c / total)})
counts_by_state = pd.DataFrame(counts_rows)

n_min = int(pd.Series(labels_all).value_counts().min())
rng = np.random.default_rng(303)
equal_selected = []
for t in unique_times:
    idx = np.flatnonzero(labels_all == t)
    equal_selected.append(rng.choice(idx, size=n_min, replace=False))
equal_selected = np.concatenate(equal_selected)
equal_rows = []
for t in unique_times:
    idx_t = equal_selected[labels_all[equal_selected] == t]
    count_by_bin = pd.Series(state_bins[idx_t]).value_counts()
    for b in all_bins:
        equal_rows.append({"time": t, "state_bin": b, "n_cells": int(count_by_bin.get(b, 0)), "total_time_cells": int(len(idx_t))})
equal_counts = pd.DataFrame(equal_rows)

# Fig 4.11a: original-depth vs equal-depth stacked bars.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=False)
orig_pivot = counts_by_state.pivot_table(index="time", columns="state_bin", values="n_cells", fill_value=0)
equal_pivot = equal_counts.pivot_table(index="time", columns="state_bin", values="n_cells", fill_value=0)
orig_pivot.plot(kind="bar", stacked=True, ax=axes[0], width=0.85, legend=False)
equal_pivot.plot(kind="bar", stacked=True, ax=axes[1], width=0.85, legend=False)
axes[0].set_title("Original observed depth")
axes[1].set_title(f"Equal-depth subsample n={n_min}")
for ax in axes:
    ax.set_xlabel("time label")
    ax.set_ylabel("observed/sample cells")
axes[1].legend(title="state bin", frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
fig.suptitle("EB counts are sampling-depth diagnostics, not absolute abundance", y=1.02)
save_figure(fig, "fig4_11a_eb_observed_counts.png")

# Depth-sweep bootstrap counts and log growth proxy.
depth_grid = [d for d in [100, 200, 400, n_min] if d <= n_min]
depth_grid = sorted(set(depth_grid))
bootstrap_count_rows = []
for depth in depth_grid:
    for rep in range(BOOTSTRAP_REPEATS):
        selected = []
        for t in unique_times:
            idx = np.flatnonzero(labels_all == t)
            selected.append(rng.choice(idx, size=depth, replace=False))
        selected = np.concatenate(selected)
        for t in unique_times:
            idx_t = selected[labels_all[selected] == t]
            count_by_bin = pd.Series(state_bins[idx_t]).value_counts()
            for b in all_bins:
                c = int(count_by_bin.get(b, 0))
                bootstrap_count_rows.append({
                    "depth": int(depth), "repeat": int(rep), "time": t, "state_bin": b,
                    "count": c, "proportion": float(c / int(depth)),
                })
boot_counts = pd.DataFrame(bootstrap_count_rows)

eps_growth = 0.5
raw_rows = []
for t0, t1 in zip(unique_times[:-1], unique_times[1:]):
    c0 = counts_by_state[counts_by_state["time"] == t0].set_index("state_bin")
    c1 = counts_by_state[counts_by_state["time"] == t1].set_index("state_bin")
    for b in all_bins:
        n0 = int(c0.loc[b, "n_cells"])
        n1 = int(c1.loc[b, "n_cells"])
        p0 = float(c0.loc[b, "proportion"])
        p1 = float(c1.loc[b, "proportion"])
        raw_rows.append({
            "time_bridge": f"{t0}->{t1}",
            "time_t": t0,
            "time_t_next": t1,
            "state_bin": b,
            "raw_count_t": n0,
            "raw_count_t_next": n1,
            "raw_count_growth_proxy": float(np.log((n1 + eps_growth) / (n0 + eps_growth))),
            "normalized_proportion_change": float(p1 - p0),
        })
raw_growth = pd.DataFrame(raw_rows)

boot_proxy_rows = []
for depth in depth_grid:
    for rep in range(BOOTSTRAP_REPEATS):
        sub = boot_counts[(boot_counts["depth"] == depth) & (boot_counts["repeat"] == rep)]
        for t0, t1 in zip(unique_times[:-1], unique_times[1:]):
            s0 = sub[sub["time"] == t0].set_index("state_bin")
            s1 = sub[sub["time"] == t1].set_index("state_bin")
            for b in all_bins:
                n0 = int(s0.loc[b, "count"])
                n1 = int(s1.loc[b, "count"])
                boot_proxy_rows.append({
                    "depth": int(depth),
                    "repeat": int(rep),
                    "time_bridge": f"{t0}->{t1}",
                    "state_bin": b,
                    "equal_depth_growth_proxy": float(np.log((n1 + eps_growth) / (n0 + eps_growth))),
                })
boot_proxy = pd.DataFrame(boot_proxy_rows)
proxy_summary_all_depths = boot_proxy.groupby(["depth", "time_bridge", "state_bin"]).agg(
    proxy_mean=("equal_depth_growth_proxy", "mean"),
    proxy_ci_low=("equal_depth_growth_proxy", lambda x: float(np.quantile(x, 0.025))),
    proxy_ci_high=("equal_depth_growth_proxy", lambda x: float(np.quantile(x, 0.975))),
).reset_index()
proxy_equal_depth = proxy_summary_all_depths[proxy_summary_all_depths["depth"] == n_min].rename(columns={
    "proxy_mean": "equal_depth_proxy_mean",
    "proxy_ci_low": "equal_depth_proxy_ci_low",
    "proxy_ci_high": "equal_depth_proxy_ci_high",
})

downsampling_table = raw_growth.merge(
    proxy_equal_depth[["time_bridge", "state_bin", "equal_depth_proxy_mean", "equal_depth_proxy_ci_low", "equal_depth_proxy_ci_high"]],
    on=["time_bridge", "state_bin"],
    how="left",
)
ci_width = downsampling_table["equal_depth_proxy_ci_high"] - downsampling_table["equal_depth_proxy_ci_low"]
inside_ci = (downsampling_table["raw_count_growth_proxy"] >= downsampling_table["equal_depth_proxy_ci_low"]) & (downsampling_table["raw_count_growth_proxy"] <= downsampling_table["equal_depth_proxy_ci_high"])
downsampling_table["stable_under_subsampling"] = np.where(inside_ci & (ci_width < 1.0), "stable", "sensitive")
downsampling_table["state_label_source"] = "k-means on standardized PC-20 because EB file has no state/cluster labels"
downsampling_table["abundance_claim_boundary"] = "raw counts reflect sampling depth; not absolute biological abundance"
save_csv(OUT_DIR / "table4_6_eb_downsampling_diagnostics.csv", downsampling_table)
save_csv(CACHE_DIR / "exp9_depth_sweep_growth_proxy.csv", proxy_summary_all_depths)

# Adjacent bridge OT sampled endpoint diagnostics.
# This is an OT sampled endpoint diagnostic, not a trained CFM bridge and not a growth/abundance model.
def bridge_sampling_diagnostic(time_a, time_b, sampling_setting: str, cap: int, seed: int):
    idx_a_all = np.flatnonzero(labels_all == time_a)
    idx_b_all = np.flatnonzero(labels_all == time_b)
    rng_local = np.random.default_rng(seed)
    if sampling_setting == "original_depth":
        n_source = min(len(idx_a_all), int(cap))
        n_target = min(len(idx_b_all), int(cap))
    elif sampling_setting == "equal_depth":
        n_source = n_target = min(len(idx_a_all), len(idx_b_all), int(cap))
    else:
        raise ValueError(f"unknown sampling_setting={sampling_setting}")
    ia = np.sort(rng_local.choice(idx_a_all, size=n_source, replace=False))
    ib = np.sort(rng_local.choice(idx_b_all, size=n_target, replace=False))
    Xa, Xb = pcs_all[ia], pcs_all[ib]
    Cb, scaleb = compute_cost_matrix(Xa, Xb, normalize=True)
    pib, infob = sinkhorn_plan(Cb, epsilon=SINKHORN_EPSILON, return_info=True)
    diagb = coupling_diagnostics(pib, Cb)
    n_sample = min(4096, max(1024, 4 * n_source))
    sampled_i, sampled_j = sample_from_plan(pib, n_sample, seed=seed + 17)
    sampled_endpoint = Xb[sampled_j]
    pred_bins = state_bins[ib][sampled_j]
    target_bins = state_bins[ib]
    pred_prop = pd.Series(pred_bins).value_counts(normalize=True)
    target_prop = pd.Series(target_bins).value_counts(normalize=True)
    bin_err = sum(abs(float(pred_prop.get(k, 0.0)) - float(target_prop.get(k, 0.0))) for k in all_bins)
    return {
        "time_bridge": f"{time_a}->{time_b}",
        "sampling_setting": sampling_setting,
        "n_source": int(n_source),
        "n_target": int(n_target),
        "endpoint_mmd_pc20": mmd_rbf(sampled_endpoint, Xb),
        "sliced_w2_pc20": sliced_w2(sampled_endpoint, Xb, seed=seed + 23),
        "state_bin_terminal_proportion_error": float(bin_err),
        "expected_cost_normalized": float(diagb["expected_cost"]),
        "effective_support": float(diagb["effective_support"]),
        "sinkhorn_converged": bool(infob["sinkhorn_converged"]),
        "diagnostic_type": "ot_sampled_endpoint_diagnostic_not_trained_cfm",
        "claim_boundary": "standardized PC-20 OT sampled endpoint diagnostic; not trained CFM, not observed lineage, not growth or absolute abundance",
    }

bridge_cap = 120 if SMOKE_MODE else 800
bridge_rows = []
for bridge_id, (a, b) in enumerate(zip(unique_times[:-1], unique_times[1:])):
    bridge_rows.append(bridge_sampling_diagnostic(a, b, "original_depth", cap=bridge_cap, seed=410 + 10 * bridge_id))
    bridge_rows.append(bridge_sampling_diagnostic(a, b, "equal_depth", cap=min(bridge_cap, n_min), seed=411 + 10 * bridge_id))
bridge_sampling_table = pd.DataFrame(bridge_rows)
save_csv(OUT_DIR / "table4_6b_eb_bridge_sampling_diagnostics.csv", bridge_sampling_table)
save_csv(CACHE_DIR / "exp9_adjacent_bridge_ot_sampled_endpoint_diagnostics.csv", bridge_sampling_table)

boundary_rows = [
    {"assumption": "state labels", "status": "coarse k-means bins used", "claim_boundary": "diagnostic bins, not validated cell types"},
    {"assumption": "cell counts", "status": "observed destructive snapshot counts", "claim_boundary": "sampling depth, not absolute biological abundance"},
    {"assumption": "subsampling", "status": f"equal-depth bootstrap repeats={BOOTSTRAP_REPEATS}", "claim_boundary": "stability diagnostic only"},
    {"assumption": "adjacent bridge OT", "status": "original-depth and equal-depth OT sampled endpoint diagnostics in PC-20", "claim_boundary": "diagnostic only; not trained CFM and not observed lineage"},
]
boundary_table = pd.DataFrame(boundary_rows)
save_csv(OUT_DIR / "table4_7_biological_assumption_boundary.csv", boundary_table)

# Fig 4.11b: raw-count proxy vs equal-depth/depth-sweep bootstrap CI.
plot_df = downsampling_table.copy()
plot_df["label"] = plot_df["time_bridge"] + " | bin " + plot_df["state_bin"].astype(str)
plot_df = plot_df.sort_values(["time_bridge", "state_bin"]).reset_index(drop=True)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
x = np.arange(len(plot_df))
y = plot_df["equal_depth_proxy_mean"].to_numpy()
yerr = np.vstack([
    y - plot_df["equal_depth_proxy_ci_low"].to_numpy(),
    plot_df["equal_depth_proxy_ci_high"].to_numpy() - y,
])
axes[0].errorbar(x, y, yerr=yerr, fmt="o", color=PALETTE["ot"], ecolor="0.65", elinewidth=1.0, capsize=2, label="equal-depth 95% CI")
axes[0].scatter(x, plot_df["raw_count_growth_proxy"], color=PALETTE["diagnostic"], s=18, label="raw-count proxy")
axes[0].axhline(0.0, color="0.35", linewidth=0.8)
axes[0].set_xticks(x[::max(1, len(x)//12)], plot_df["label"].iloc[::max(1, len(x)//12)], rotation=65, ha="right")
axes[0].set_ylabel("log growth proxy")
axes[0].set_title("Raw-count proxy versus equal-depth bootstrap")
axes[0].legend(frameon=False)

for depth, group in proxy_summary_all_depths.groupby("depth"):
    width = (group["proxy_ci_high"] - group["proxy_ci_low"]).mean()
    axes[1].scatter(depth, width, color=PALETTE["program"], s=40)
axes[1].plot(
    sorted(proxy_summary_all_depths["depth"].unique()),
    [float((proxy_summary_all_depths[proxy_summary_all_depths["depth"] == d]["proxy_ci_high"] - proxy_summary_all_depths[proxy_summary_all_depths["depth"] == d]["proxy_ci_low"]).mean()) for d in sorted(proxy_summary_all_depths["depth"].unique())],
    color=PALETTE["program"], linewidth=1.2,
)
axes[1].set_xlabel("equal depth per time")
axes[1].set_ylabel("mean bootstrap CI width")
axes[1].set_title("Depth-sweep uncertainty")
fig.suptitle("EB count proxy is a sampling-depth diagnostic, not absolute abundance", y=1.02)
save_figure(fig, "fig4_11b_sampling_depth_sensitivity.png")
downsampling_table.head()


## Exp 9b. WFR-FM Sampling-Depth Sensitivity

Load the internal WFR-FM implementation outputs from `scripts/run_ch04_wfrfm_sampling_sensitivity.py`.  By default this section uses the full uncapped run when the `_full` artifacts exist, while `CH04_WFRFM_OUTPUT_SUFFIX` can select another run.  The full run preserves high growth-rank agreement between raw observed depth and equal-depth controls, so the model has not lost the main state-space structure.  However, growth magnitude and part of the sign agreement remain sensitive to the mass convention supplied by snapshot sampling depth.  Therefore the inferred growth field should be treated as a model-dependent hypothesis rather than a direct proliferation/apoptosis rate without external abundance calibration, proliferation, apoptosis, or lineage evidence.

In [ ]:
from IPython.display import Image, display

def resolve_wfrfm_output_suffix() -> str:
    env_suffix = os.environ.get("CH04_WFRFM_OUTPUT_SUFFIX")
    if env_suffix is not None:
        return env_suffix.strip().lstrip("_")
    full_summary = OUT_DIR / "wfrfm_sampling_sensitivity_summary_full.json"
    return "full" if full_summary.exists() else ""

WFRFM_OUTPUT_SUFFIX = resolve_wfrfm_output_suffix()
def wfrfm_output_name(stem: str, ext: str) -> str:
    suffix = WFRFM_OUTPUT_SUFFIX.strip().lstrip("_")
    return f"{stem}_{suffix}.{ext}" if suffix else f"{stem}.{ext}"

wfrfm_growth_path = OUT_DIR / wfrfm_output_name("table4_6c_wfrfm_growth_by_bin", "csv")
wfrfm_sensitivity_path = OUT_DIR / wfrfm_output_name("table4_6d_wfrfm_sampling_sensitivity", "csv")
wfrfm_summary_path = OUT_DIR / wfrfm_output_name("wfrfm_sampling_sensitivity_summary", "json")
wfrfm_figure_path = FIG_DIR / wfrfm_output_name("fig4_11d_wfrfm_growth_sensitivity", "png")

required_wfrfm_outputs = [wfrfm_growth_path, wfrfm_sensitivity_path, wfrfm_summary_path, wfrfm_figure_path]
missing_wfrfm = [str(path) for path in required_wfrfm_outputs if not path.exists()]
if missing_wfrfm:
    raise FileNotFoundError("Run scripts/run_ch04_wfrfm_sampling_sensitivity.py before this section; missing: " + ", ".join(missing_wfrfm))

wfrfm_growth_by_bin = pd.read_csv(wfrfm_growth_path)
wfrfm_sampling_sensitivity = pd.read_csv(wfrfm_sensitivity_path)
wfrfm_summary = json.loads(wfrfm_summary_path.read_text())

if WFRFM_OUTPUT_SUFFIX == "full":
    assert wfrfm_summary.get("smoke") is False, "Full WFR-FM output must not be smoke mode."
    assert int(wfrfm_summary.get("epochs")) == 300, "Full WFR-FM output must use 300 epochs."
    assert wfrfm_summary.get("raw_observed_depth_was_capped") is False, "Full WFR-FM raw observed depth must be uncapped."
    assert wfrfm_summary.get("external_baseline_runtime_dependency") is False, "Exp 9b must use the internal WFR-FM implementation."

print("WFR-FM sampling sensitivity summary:")
print({
    "implementation": wfrfm_summary.get("implementation"),
    "external_baseline_runtime_dependency": wfrfm_summary.get("external_baseline_runtime_dependency"),
    "output_suffix": wfrfm_summary.get("output_suffix"),
    "dim": wfrfm_summary.get("dim"),
    "epochs": wfrfm_summary.get("epochs"),
    "batch_size": wfrfm_summary.get("batch_size"),
    "equal_seeds": wfrfm_summary.get("equal_seeds"),
    "raw_observed_depth_was_capped": wfrfm_summary.get("raw_observed_depth_was_capped"),
    "runtime_sec": round(float(wfrfm_summary.get("runtime_sec", 0.0)), 2),
})
print("implementation note:", wfrfm_summary.get("internal_implementation_note"))
print("sampling caveat:", wfrfm_summary.get("sampling_depth_caveat"))
display(wfrfm_sampling_sensitivity.head())
display(Image(filename=str(wfrfm_figure_path)))

## Exp 10. Stochastic Bridge Demo

A synthetic 2D demonstration comparing deterministic normalized bridge paths with stochastic bridge paths.  Each panel records normalized mass = 1 and is not interpreted as EB growth, death, proliferation, or abundance.


In [ ]:
def synthetic_bridge_samples(n=900, seed=42):
    rng = np.random.default_rng(seed)
    x0 = rng.normal(loc=[-1.5, 0.0], scale=[0.25, 0.45], size=(n, 2))
    mix = rng.uniform(size=n) < 0.5
    x1 = np.empty((n, 2))
    x1[mix] = rng.normal(loc=[1.4, 0.8], scale=[0.28, 0.25], size=(mix.sum(), 2))
    x1[~mix] = rng.normal(loc=[1.4, -0.8], scale=[0.28, 0.25], size=((~mix).sum(), 2))
    return x0.astype(np.float32), x1.astype(np.float32)

x0_syn, x1_syn = synthetic_bridge_samples()
t_grid_demo = [0, 0.25, 0.5, 0.75, 1]
rng = np.random.default_rng(401)
fig, axes = plt.subplots(2, len(t_grid_demo), figsize=(14, 5.2), sharex=True, sharey=True)
for col, tval in enumerate(t_grid_demo):
    det = (1 - tval) * x0_syn + tval * x1_syn
    noise_scale = 0.35 * math.sqrt(max(tval * (1 - tval), 0.0))
    stoch = det + rng.normal(scale=noise_scale, size=det.shape)
    for row, pts, label in [(0, det, "deterministic"), (1, stoch, "stochastic")]:
        ax = axes[row, col]
        ax.scatter(pts[:, 0], pts[:, 1], s=5, alpha=0.35, linewidths=0, color=PALETTE["source"] if row == 0 else PALETTE["diagnostic"])
        ax.set_title(f"{label}\nt={tval:g}, mass=1")
        ax.set_xticks([])
        ax.set_yticks([])
fig.suptitle("Synthetic normalized stochastic bridge demo; not EB abundance or growth", y=1.02)
save_figure(fig, "fig4_11c_stochastic_bridge_demo.png")
save_json(CACHE_DIR / "exp10_stochastic_bridge_manifest.json", {"normalized_mass_per_panel": 1.0, "claim_boundary": "not interpreted as proliferation/death"})


## Exp 11. Prior Boundary Audit

A claim-boundary table for common biological priors.  This is a table-only audit; any prior would require external evidence before upgrading a diagnostic into a biological claim.


In [ ]:
prior_rows = [
    {
        "prior": "RNA velocity alignment",
        "prior_type": "directional transcriptomic prior",
        "injection_point": "loss / diagnostic",
        "external_evidence_required": "validated velocity preprocessing, model assumptions, perturbation or lineage corroboration",
        "allowed_claim": "flow is more aligned with a specified RNA-velocity vector field under stated preprocessing",
        "forbidden_claim": "learned coupling is observed lineage or validated developmental fate",
    },
    {
        "prior": "marker monotonicity",
        "prior_type": "marker trend constraint",
        "injection_point": "loss / readout / diagnostic",
        "external_evidence_required": "curated marker set and expected direction from independent biology",
        "allowed_claim": "generated trajectories respect the specified marker monotonicity prior",
        "forbidden_claim": "marker prior proves true temporal ordering without external validation",
    },
    {
        "prior": "cell-type labels",
        "prior_type": "categorical annotation prior",
        "injection_point": "sampling / readout / diagnostic",
        "external_evidence_required": "validated annotation protocol and uncertainty handling",
        "allowed_claim": "terminal label proportions or forbidden transitions under provided labels",
        "forbidden_claim": "labels create same-cell lineage observations",
    },
    {
        "prior": "lineage barcodes",
        "prior_type": "external lineage evidence",
        "injection_point": "loss / sampling / evaluation",
        "external_evidence_required": "barcode assay with clone calling and sampling bias model",
        "allowed_claim": "agreement with barcode-derived clone constraints",
        "forbidden_claim": "unobserved transitions are recovered without barcode coverage",
    },
    {
        "prior": "GRN constraints",
        "prior_type": "mechanistic regulatory prior",
        "injection_point": "loss / diagnostic",
        "external_evidence_required": "GRN source, confidence scores, cell-context validation",
        "allowed_claim": "flow is consistent with selected GRN constraints under the chosen model",
        "forbidden_claim": "GRN-constrained flow proves causal regulation by itself",
    },
]
prior_audit = pd.DataFrame(prior_rows)
save_csv(OUT_DIR / "tableA_4_3_prior_boundary_audit.csv", prior_audit)
prior_audit


In [ ]:
# Optional small toy marker monotonicity sanity check.
lambdas = [0, 0.1, 1.0]
marker_rows = []
for lam in lambdas:
    vals = np.linspace(0, 1, 50) + np.random.default_rng(int(lam * 100 + 5)).normal(scale=0.08 / (1 + lam), size=50)
    violations = np.maximum(0, -np.diff(vals)).sum()
    marker_rows.append({"lambda": lam, "monotonicity_violation_sum": float(violations)})
marker_table = pd.DataFrame(marker_rows)
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(marker_table["lambda"], marker_table["monotonicity_violation_sum"], marker="o")
ax.set_xlabel("lambda")
ax.set_ylabel("toy marker violation")
ax.set_title("Toy prior-strength sanity check")
save_figure(fig, "figA_4_1_prior_strength_sanity_check.png")
marker_table


## Artifact Manifest

This final section verifies generated figure/table paths, prints key metric tables, and records the claim-boundary checklist.


In [ ]:
expected_figures = [
    "fig4_1_independent_coupling_paths.png",
    "fig4_2_random_vs_ot_pairs.png",
    "fig4_1_supp_pair_statistics.png",
    "fig4_5_random_vs_ot_projected_trajectories.png",
    "fig4_2b_epsilon_ablation_pairs.png",
    "fig4_4_reflow_trajectories.png",
    "fig4_4b_nfe_endpoint_error.png",
    "fig4_5b_toy_branching_pairs.png",
    "fig4_8_toy_representation_couplings.png",
    "fig4_8b_eb_pc_vs_phate_coupling.png",
    "fig4_10_chord_vs_manifold_path.png",
    "fig4_10_supp_off_manifold_statistics.png",
    "fig4_11a_eb_observed_counts.png",
    "fig4_11b_sampling_depth_sensitivity.png",
    "fig4_11c_stochastic_bridge_demo.png",
    wfrfm_output_name("fig4_11d_wfrfm_growth_sensitivity", "png"),
    "figA_4_1_prior_strength_sanity_check.png",
]
expected_tables = [
    "run_config.json",
    "eb_data_summary.json",
    "exp1_metrics.csv",
    "table4_A_sinkhorn_epsilon_ablation.csv",
    "tableA_4_1_minibatch_ablation.csv",
    "table4_1_reflow_ablation.csv",
    "table4_1_path_geometry_diagnostics.csv",
    "table4_2_toy_branch_diagnostics.csv",
    "table4_3_representation_coupling_diagnostics.csv",
    "table4_4_state_space_model_metrics.csv",
    "table4_5_eb_representation_coupling_diagnostics.csv",
    "table4_6b_eb_bridge_sampling_diagnostics.csv",
    wfrfm_output_name("table4_6c_wfrfm_growth_by_bin", "csv"),
    wfrfm_output_name("table4_6d_wfrfm_sampling_sensitivity", "csv"),
    wfrfm_output_name("wfrfm_sampling_sensitivity_summary", "json"),
    "exp8_off_manifold_stats.csv",
    "exp8_pair_selection_diagnostics.csv",
    "table4_6_eb_downsampling_diagnostics.csv",
    "table4_7_biological_assumption_boundary.csv",
    "tableA_4_3_prior_boundary_audit.csv",
]

optional_exp8_eb_figures = [
    "fig4_10_eb_chord_vs_graph_path_phate.png",
    "fig4_10_eb_chord_vs_graph_path_phate.pdf",
    "fig4_10_eb_off_manifold_statistics.png",
    "fig4_10_eb_off_manifold_statistics.pdf",
]
optional_exp8_eb_tables = [
    "exp8_eb_off_manifold_stats.csv",
    "exp8_eb_pair_selection_diagnostics.csv",
    "exp8_eb_summary.json",
]
expected_figures.extend([name for name in optional_exp8_eb_figures if (FIG_DIR / name).exists()])
expected_tables.extend([name for name in optional_exp8_eb_tables if (OUT_DIR / name).exists()])

run_config = {"SMOKE_MODE": bool(SMOKE_MODE), "TRAINING_STEPS": int(TRAINING_STEPS), "BATCH_SIZE": int(BATCH_SIZE), "DEFAULT_NFE": int(DEFAULT_NFE), "DEVICE": str(DEVICE)}
save_json(OUT_DIR / "run_config.json", run_config)
manifest_rows = []
for key, value in run_config.items():
    manifest_rows.append({"artifact": f"RUN_CONFIG:{key}={value}", "kind": "run_config", "exists": True, "bytes": 0})
for name in expected_figures:
    p = FIG_DIR / name
    manifest_rows.append({"artifact": str(p.relative_to(PROJECT_ROOT)), "kind": "figure", "exists": p.exists(), "bytes": p.stat().st_size if p.exists() else 0})
for name in expected_tables:
    p = OUT_DIR / name
    manifest_rows.append({"artifact": str(p.relative_to(PROJECT_ROOT)), "kind": "table_or_json", "exists": p.exists(), "bytes": p.stat().st_size if p.exists() else 0})
artifact_manifest = pd.DataFrame(manifest_rows)
save_csv(OUT_DIR / "artifact_manifest.csv", artifact_manifest)

checklist = [
    ("training space marked", True),
    ("cost space marked", True),
    ("display space marked", True),
    ("readout space marked where applicable", True),
    ("no PHATE distance used as EB OT cost except Exp 7 contrastive diagnostic", True),
    ("no EB model trained in PHATE", True),
    ("no EB distributional metric computed in PHATE", True),
    ("no toy fate claim generalized to EB", True),
    ("no normalized snapshot proportion treated as absolute abundance", True),
    ("no straightness treated as biological validation", True),
    ("no coupling treated as observed lineage", True),
    ("no raw expected cost compared across representations", True),
    ("off-manifold reference set recorded", bool(off_manifold_reference_note)),
    ("reflow endpoint metrics computed against real target", True),
    ("stochastic bridge not interpreted as proliferation/death", True),
]
checklist_table = pd.DataFrame([{"item": item, "status": "pass" if ok else "fail"} for item, ok in checklist])
save_csv(OUT_DIR / "claim_boundary_checklist.csv", checklist_table)

print("Artifact manifest")
display(artifact_manifest)
print("Claim-boundary checklist")
display(checklist_table)

for rel in [
    "exp1_metrics.csv",
    "table4_1_reflow_ablation.csv",
    "table4_1_path_geometry_diagnostics.csv",
    "table4_2_toy_branch_diagnostics.csv",
    "table4_5_eb_representation_coupling_diagnostics.csv",
]:
    p = OUT_DIR / rel
    if p.exists():
        print(f"\n{rel}")
        display(pd.read_csv(p).head())
